In [2]:
"""
OceanXRay — Tier 1: FFT-based wavelength & propagation-direction extraction
=============================================================================
For each land-filtered internal-wave detection in
real_internal_wave_detections.parquet (written by infer_iw_on_real_scenes.py):

  1. Load the source SAR tile the detection came from.
  2. Crop a padded window around the detection box (`box_resized`,
     [x1, y1, x2, y2] in the 1024x1024 tile's pixel space).
  3. Apply a 2D Hann window (suppresses edge-discontinuity leakage).
  4. 2D FFT -> dominant spatial-frequency peak (DC excluded).
  5. Convert the peak to:
       - wavelength in meters (tile pixel scale ~24.4 m/px)
       - propagation direction, first in IMAGE space, then converted to a
         real GEOGRAPHIC bearing (0=N, 90=E) using the scene's own GCP
         transform. Sentinel-1 GRD images are in radar (slant/ground-range)
         geometry, NOT north-up -- an image-space "up" is the satellite's
         flight direction, which is rotated ~10-15 deg off north and
         flipped between ascending and descending passes. Using image-space
         bearings directly would aim every Tier 2 ray in the wrong direction.
       - Both 180-deg-apart candidates are kept (bearing_deg_a, bearing_deg_b):
         a real-valued FFT is Hermitian-symmetric, so it physically cannot
         tell "this way" from "opposite way". Tier 2 resolves it.

HOW TO RUN
----------
Kaggle/Colab notebook cell (paste or %run): set MODE below to "selftest"
first, run it, then set MODE = "run" and run again.

Terminal:
    python tier1_fft_backtrack.py selftest
    python tier1_fft_backtrack.py run

Output: tier1_fft_results.parquet
"""

import sys
import glob
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

# ======================= EDIT THESE =======================
MODE = "run"   # "selftest" first, then "run" (used when run in a notebook)

DETECTIONS_PARQUET = "/kaggle/input/datasets/mayankgupta005/internal-wave-detection-metadata/real_internal_wave_detections.parquet"
TILES_DIR = "/kaggle/input/datasets/mayankgupta005/sar-image-tiles-june-2021/sar_images_for_iw_detection"          # same folder as IMAGE_DIR in infer_iw_on_real_scenes.py
SAFE_ROOT = "/kaggle/input/datasets/andrejkarpathy/sentinel-1-june-2021-raw-safe"       # same folder as SAFE_ROOT in infer_iw_on_real_scenes.py
OUTPUT_PARQUET = "tier1_fft_results.parquet"
LAND_MASK_PATH = "/kaggle/input/datasets/mayankgupta005/internal-wave-detection-metadata/land_ocean_mask.npz"  # same file infer_iw_on_real_scenes.py uses

# These two are confirmed from convert_scenes_to_images.py
# (TILE_SIZE_KM = 25, OUTPUT_PIXELS = 1024) -- no need to change them
# unless you changed those settings there.
TILE_SIZE_KM = 25
OUTPUT_PIXELS = 1024
# ==========================================================

NATIVE_PX_M = 10.0                                    # S1 IW GRD pixel spacing
NATIVE_TILE_PX = int(TILE_SIZE_KM * 1000 / NATIVE_PX_M)  # 2500
TILE_TO_SCENE = NATIVE_TILE_PX / OUTPUT_PIXELS        # 1 tile px = 2.44 scene px
PIXEL_SCALE_M = TILE_SIZE_KM * 1000 / OUTPUT_PIXELS   # ~24.4 m per tile px

CROP_PAD_PX = 40
MIN_CROP_SIDE_PX = 48
DC_EXCLUSION_RADIUS = 2
LOW_SNR_THRESHOLD = 3.0

M_PER_DEG_LAT = 110_540.0
M_PER_DEG_LON_EQ = 111_320.0


# ----------------------------------------------------------------------
# Core FFT routine
# ----------------------------------------------------------------------
def image_bearing_from_freq(fy, fx):
    """Bearing in IMAGE space (0 = image-up, 90 = image-right)."""
    return np.degrees(np.arctan2(fx, -fy)) % 360.0


def extract_dominant_peak(crop, pixel_scale_m):
    """Window -> FFT -> peak. Returns dict (incl. raw freq_y/freq_x) or None."""
    h, w = crop.shape
    if h < MIN_CROP_SIDE_PX or w < MIN_CROP_SIDE_PX:
        return None

    crop = crop.astype(np.float64)
    crop = crop - crop.mean()
    windowed = crop * np.outer(np.hanning(h), np.hanning(w))

    power = np.abs(np.fft.fftshift(np.fft.fft2(windowed))) ** 2

    cy, cx = h // 2, w // 2
    yy, xx = np.mgrid[0:h, 0:w]
    search_mask = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2) > DC_EXCLUSION_RADIUS
    if not search_mask.any():
        return None

    masked = np.where(search_mask, power, -np.inf)
    peak_y, peak_x = np.unravel_index(np.argmax(masked), masked.shape)

    freq_y = (peak_y - cy) / h
    freq_x = (peak_x - cx) / w
    freq_mag = np.hypot(freq_y, freq_x)
    if freq_mag <= 0:
        return None

    median_bg = np.median(power[search_mask])
    peak_snr = float(power[peak_y, peak_x] / median_bg) if median_bg > 0 else np.nan

    return {
        "freq_y": float(freq_y),
        "freq_x": float(freq_x),
        "wavelength_m": float(pixel_scale_m / freq_mag),
        "image_bearing_deg": float(image_bearing_from_freq(freq_y, freq_x)),
        "spectral_peak_snr": peak_snr,
    }


# ----------------------------------------------------------------------
# Image-space -> geographic bearing via the scene's GCP transform
# ----------------------------------------------------------------------
def geographic_bearing(transform, cx_scene, cy_scene, freq_y, freq_x, half_len_px=100.0):
    """
    Convert an image-space wavevector to a geographic propagation bearing.

    We transform the CREST direction (perpendicular to the wavevector),
    not the wavevector itself: a crest line is a physical displacement
    and maps correctly through the pixel->lon/lat transform, whereas a
    normal vector does not under a non-conformal mapping (lon/lat degrees
    are not equal-length). The propagation bearing is then the crest
    tangent rotated 90 deg, measured in local east/north METERS.
    Returns (bearing_a, bearing_b) -- 180 deg apart.
    """
    k = np.array([freq_x, freq_y])          # (x=col, y=row)
    k = k / np.linalg.norm(k)
    t = np.array([-k[1], k[0]])             # crest tangent in pixel space

    p1 = (cx_scene - half_len_px * t[0], cy_scene - half_len_px * t[1])
    p2 = (cx_scene + half_len_px * t[0], cy_scene + half_len_px * t[1])
    lon1, lat1 = transform * p1
    lon2, lat2 = transform * p2

    mean_lat = np.radians((lat1 + lat2) / 2)
    east = (lon2 - lon1) * M_PER_DEG_LON_EQ * np.cos(mean_lat)
    north = (lat2 - lat1) * M_PER_DEG_LAT

    crest_bearing = np.degrees(np.arctan2(east, north)) % 360.0
    bearing_a = (crest_bearing + 90.0) % 360.0
    bearing_b = (crest_bearing + 270.0) % 360.0
    return float(bearing_a), float(bearing_b)


# ----------------------------------------------------------------------
# Scene transform loading (same method as infer_iw_on_real_scenes.py)
# ----------------------------------------------------------------------
def find_measurement_tiff(safe_dir, polarization="vv"):
    matches = glob.glob(os.path.join(safe_dir, "measurement", f"*{polarization}*.tiff"))
    if not matches:
        matches = glob.glob(os.path.join(safe_dir, "**", "measurement",
                                         f"*{polarization}*.tiff"), recursive=True)
    if not matches:
        raise FileNotFoundError(f"No {polarization} measurement GeoTIFF under {safe_dir}")
    return matches[0]


def load_scene_transform(scene_id):
    import rasterio
    from rasterio.transform import from_gcps
    tiff = find_measurement_tiff(os.path.join(SAFE_ROOT, f"{scene_id}.SAFE"))
    with rasterio.open(tiff) as src:
        gcps, _ = src.get_gcps()
        if not gcps:
            raise ValueError(f"No GCPs in {tiff}")
        return from_gcps(gcps)


# ----------------------------------------------------------------------
# Tile-filename lookup + land mask (for parquets from older inference runs)
# ----------------------------------------------------------------------
TILE_FILENAME_RE = re.compile(r"^(.+)_tile(\d+)_row(\d+)_col(\d+)\.png$")


def build_tile_lookup(tiles_dir):
    """Map (scene_id, tile_idx) -> tile filename by scanning TILES_DIR.
    Older parquets only store scene_id + tile_idx, not the filename."""
    lookup = {}
    for p in Path(tiles_dir).glob("*.png"):
        m = TILE_FILENAME_RE.match(p.name)
        if m:
            lookup[(m.group(1), int(m.group(2)))] = p.name
    return lookup


def load_land_checker(path):
    """Same lookup as infer_iw_on_real_scenes.py's is_on_land."""
    d = np.load(path)
    mask, mlat, mlon = d["mask"], d["lat"], d["lon"]

    def is_on_land(lat, lon):
        lat = max(mlat.min(), min(mlat.max(), lat))
        lon = max(mlon.min(), min(mlon.max(), lon))
        i = int((lat - mlat[0]) / (mlat[1] - mlat[0]))
        j = int((lon - mlon[0]) / (mlon[1] - mlon[0]))
        return not bool(mask[i, j])   # mask True = ocean
    return is_on_land


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------
def get_box_xyxy(row):
    """Read the detection box as [x1, y1, x2, y2] in 1024-tile pixels."""
    if "box_resized" in row.index and row["box_resized"] is not None:
        b = np.asarray(row["box_resized"], dtype=float)
        return b[0], b[1], b[2], b[3]
    if {"bbox_x", "bbox_y", "bbox_w", "bbox_h"}.issubset(row.index):
        return (row["bbox_x"], row["bbox_y"],
                row["bbox_x"] + row["bbox_w"], row["bbox_y"] + row["bbox_h"])
    raise KeyError("No box column found (expected 'box_resized')")


def crop_box(tile, x1, y1, x2, y2, pad):
    h, w = tile.shape
    return tile[max(0, int(y1) - pad):min(h, int(y2) + pad),
                max(0, int(x1) - pad):min(w, int(x2) + pad)]


def angular_diff(a, b):
    d = abs(a - b) % 360.0
    return min(d, 360.0 - d)


def make_synthetic_stripe(h, w, wavelength_px, image_bearing_deg, noise_std=0.05, seed=0):
    rng = np.random.default_rng(seed)
    fx = np.sin(np.radians(image_bearing_deg))
    fy = -np.cos(np.radians(image_bearing_deg))
    yy, xx = np.mgrid[0:h, 0:w]
    img = np.sin(2 * np.pi * (fx * xx + fy * yy) / wavelength_px)
    return img + rng.normal(0, noise_std, size=img.shape)


# ----------------------------------------------------------------------
# Self-test
# ----------------------------------------------------------------------
def selftest():
    from affine import Affine
    print("=" * 70)
    print("TIER 1 SELF-TEST")
    print("=" * 70)
    ok_all = True

    print("\n[1] Wavelength + image-space bearing on synthetic stripes")
    for wl, brg in [(30, 0), (30, 90), (45, 135), (60, 200), (20, 315)]:
        r = extract_dominant_peak(make_synthetic_stripe(256, 256, wl, brg), 1.0)
        b_a = r["image_bearing_deg"]
        ok = (abs(r["wavelength_m"] - wl) < 3.5
              and min(angular_diff(b_a, brg), angular_diff(b_a + 180, brg)) < 10)
        ok_all &= ok
        print(f"  {'PASS' if ok else 'FAIL'}  true {wl}px @ {brg}deg -> "
              f"{r['wavelength_m']:.1f}px @ {b_a:.1f}/{(b_a+180)%360:.1f}deg")

    print("\n[2] Image -> geographic bearing through a rotated GCP-style transform")
    # Build pixel->lon/lat transforms like a real S1 scene: 10 m pixels,
    # image rotated by `rot` deg relative to north, at lat ~12N.
    lat0, lon0 = 12.0, 90.0
    dlat = NATIVE_PX_M / M_PER_DEG_LAT
    dlon = NATIVE_PX_M / (M_PER_DEG_LON_EQ * np.cos(np.radians(lat0)))
    for rot in [0.0, -12.0, 15.0, 168.0]:   # ~ascending, descending-like headings
        th = np.radians(rot)
        # pixel (x right, y down) -> (east, north) meters, rotated clockwise by rot:
        #   east  =  cos(th)*x - sin(th)*y
        #   north = -sin(th)*x - cos(th)*y
        # (a vector at image bearing b then lands at geographic bearing b+rot)
        a_e, b_e = np.cos(th) * NATIVE_PX_M, -np.sin(th) * NATIVE_PX_M
        d_n, e_n = -np.sin(th) * NATIVE_PX_M, -np.cos(th) * NATIVE_PX_M
        T = Affine(a_e / NATIVE_PX_M * dlon, b_e / NATIVE_PX_M * dlon, lon0,
                   d_n / NATIVE_PX_M * dlat, e_n / NATIVE_PX_M * dlat, lat0)
        for img_brg in [0.0, 70.0, 225.0]:
            r = extract_dominant_peak(make_synthetic_stripe(256, 256, 40, img_brg), 1.0)
            ga, gb = geographic_bearing(T, 1000, 1000, r["freq_y"], r["freq_x"])
            expected = (img_brg + rot) % 360.0
            ok = min(angular_diff(ga, expected), angular_diff(gb, expected)) < 10
            ok_all &= ok
            print(f"  {'PASS' if ok else 'FAIL'}  image rot {rot:+6.1f}deg, image bearing "
                  f"{img_brg:5.1f} -> geo {ga:5.1f}/{gb:5.1f} (expected {expected:5.1f})")

    print("\n" + "=" * 70)
    print("ALL SELF-TESTS PASSED." if ok_all else "SELF-TEST FAILURES -- do not run on real data.")
    print("=" * 70)
    return ok_all


# ----------------------------------------------------------------------
# Real run
# ----------------------------------------------------------------------
def run():
    det_path, tiles_dir = Path(DETECTIONS_PARQUET), Path(TILES_DIR)
    for p, what in [(det_path, "DETECTIONS_PARQUET"), (tiles_dir, "TILES_DIR"),
                    (Path(SAFE_ROOT), "SAFE_ROOT")]:
        if not p.exists():
            print(f"ERROR: {what} not found: {p}")
            return

    det = pd.read_parquet(det_path)
    print(f"Loaded {len(det)} detections. Columns: {list(det.columns)}")
    print(f"Tile scale: {PIXEL_SCALE_M:.2f} m/px "
          f"({TILE_SIZE_KM} km tiles resampled to {OUTPUT_PIXELS}px)")

    # Land mask: older parquets were written before land-masking existed
    # (the original 798-detection run had ~450 detections on land). Always
    # re-apply it here so Tier 1 never measures rivers/deltas.
    if not Path(LAND_MASK_PATH).exists():
        print(f"ERROR: LAND_MASK_PATH not found: {LAND_MASK_PATH}")
        return
    is_on_land = load_land_checker(LAND_MASK_PATH)
    on_land = det.apply(lambda r: is_on_land(float(r["real_lat"]), float(r["real_lon"])), axis=1)
    print(f"Land mask: dropping {int(on_land.sum())} of {len(det)} detections on land")
    det = det[~on_land].reset_index(drop=True)

    # Tile filename: present in newer parquets, rebuilt from scene_id + tile_idx otherwise
    if "tile_filename" not in det.columns:
        lookup = build_tile_lookup(tiles_dir)
        print(f"No 'tile_filename' column -- matched via scene_id + tile_idx "
              f"against {len(lookup)} tiles in TILES_DIR")
        det["tile_filename"] = [lookup.get((s, int(t))) for s, t in
                                zip(det["scene_id"], det["tile_idx"])]

    tile_cache, transform_cache = {}, {}
    results = []
    n_missing_tile = n_bad_crop = n_no_transform = n_low = 0

    for i, (_, row) in enumerate(det.iterrows()):
        fname = row["tile_filename"]
        if fname is None:
            n_missing_tile += 1
            continue
        if fname not in tile_cache:
            tp = tiles_dir / fname
            tile_cache[fname] = np.array(Image.open(tp).convert("L")) if tp.exists() else None
        tile = tile_cache[fname]
        if tile is None:
            n_missing_tile += 1
            continue

        x1, y1, x2, y2 = get_box_xyxy(row)
        crop = crop_box(tile, x1, y1, x2, y2, CROP_PAD_PX)
        peak = extract_dominant_peak(crop, PIXEL_SCALE_M)
        if peak is None:
            n_bad_crop += 1
            continue

        scene_id = row["scene_id"]
        if scene_id not in transform_cache:
            try:
                transform_cache[scene_id] = load_scene_transform(scene_id)
            except Exception as e:
                print(f"  WARNING: no GCP transform for {scene_id}: {e}")
                transform_cache[scene_id] = None
        T = transform_cache[scene_id]

        if T is not None:
            geo_a, geo_b = geographic_bearing(
                T, row["scene_pixel_x"], row["scene_pixel_y"],
                peak["freq_y"], peak["freq_x"],
            )
        else:
            geo_a = geo_b = np.nan
            n_no_transform += 1

        low = (not np.isfinite(peak["spectral_peak_snr"])
               or peak["spectral_peak_snr"] < LOW_SNR_THRESHOLD
               or T is None)
        n_low += int(low)

        results.append({
            "scene_id": scene_id,
            "tile_filename": fname,
            "real_lat": row.get("real_lat"),
            "real_lon": row.get("real_lon"),
            "confidence": row.get("confidence"),
            "crop_h": crop.shape[0], "crop_w": crop.shape[1],
            "wavelength_m": peak["wavelength_m"],
            "image_bearing_deg": peak["image_bearing_deg"],
            "bearing_deg_a": geo_a,
            "bearing_deg_b": geo_b,
            "spectral_peak_snr": peak["spectral_peak_snr"],
            "low_confidence": low,
        })
        if (i + 1) % 100 == 0:
            print(f"  processed {i + 1}/{len(det)}")

    out = pd.DataFrame(results)
    out.to_parquet(OUTPUT_PARQUET, index=False)

    print(f"\n{'=' * 60}\nTIER 1 SUMMARY\n{'=' * 60}")
    print(f"Detections in:              {len(det)}")
    print(f"Processed:                  {len(out)}")
    print(f"Tile file not found:        {n_missing_tile}")
    print(f"Crop too small/degenerate:  {n_bad_crop}")
    print(f"No GCP transform (bearing NaN): {n_no_transform}")
    print(f"Low-confidence flagged:     {n_low} ({100 * n_low / max(1, len(out)):.1f}%)")
    if len(out):
        print("\nWavelength (m) distribution:")
        print(out["wavelength_m"].describe().round(1).to_string())
        print("\nSpectral peak SNR distribution:")
        print(out["spectral_peak_snr"].describe().round(1).to_string())
    print(f"\nSaved to {OUTPUT_PARQUET}")


def main():
    # In a notebook, sys.argv holds Jupyter's own kernel args -- ignore them
    # and use MODE. From a terminal, a 'selftest'/'run' argument wins.
    in_notebook = "ipykernel" in sys.modules
    mode = MODE
    if not in_notebook and len(sys.argv) == 2 and sys.argv[1] in ("selftest", "run"):
        mode = sys.argv[1]
    print(f"Mode: {mode}")
    if mode == "selftest":
        selftest()
    elif mode == "run":
        run()
    else:
        print(f"Unknown MODE {mode!r}; use 'selftest' or 'run'.")


if __name__ == "__main__":
    main()

Mode: run
Loaded 798 detections. Columns: ['scene_id', 'tile_idx', 'confidence', 'box_resized', 'scene_pixel_x', 'scene_pixel_y', 'real_lon', 'real_lat']
Tile scale: 24.41 m/px (25 km tiles resampled to 1024px)
Land mask: dropping 452 of 798 detections on land
No 'tile_filename' column -- matched via scene_id + tile_idx against 4524 tiles in TILES_DIR
  processed 100/346
  processed 200/346
  processed 300/346

TIER 1 SUMMARY
Detections in:              346
Processed:                  346
Tile file not found:        0
Crop too small/degenerate:  0
No GCP transform (bearing NaN): 0
Low-confidence flagged:     0 (0.0%)

Wavelength (m) distribution:
count      346.0
mean      5196.3
std       2302.5
min        166.9
25%       3555.9
50%       5068.6
75%       6841.7
max      10795.7

Spectral peak SNR distribution:
count       346.0
mean      45116.7
std       79898.2
min         139.3
25%        1180.0
50%        5118.1
75%       49418.0
max      414096.8

Saved to tier1_fft_results.parq

In [3]:
""" V2
OceanXRay — Tier 1: FFT-based wavelength & propagation-direction extraction
=============================================================================
For each land-filtered internal-wave detection in
real_internal_wave_detections.parquet (written by infer_iw_on_real_scenes.py):

  1. Load the source SAR tile the detection came from.
  2. Crop a padded window around the detection box (`box_resized`,
     [x1, y1, x2, y2] in the 1024x1024 tile's pixel space).
  3. Remove the large-scale brightness plane, apply a 2D Hann window.
  4. 2D FFT, divide by a fitted red-noise background, and take the strongest
     directional peak among wavelengths 250 m .. crop/3 (with a harmonic
     check). v1 skipped the background step and on real data latched onto
     crop-scale brightness trends instead of soliton spacing -- see
     extract_dominant_peak.
     Detections are flagged low_confidence unless SNR >= 20 (above the
     99th percentile of wave-free crops) and >= 5 periods fit in the crop.
  5. Convert the peak to:
       - wavelength in meters (tile pixel scale ~24.4 m/px)
       - propagation direction, first in IMAGE space, then converted to a
         real GEOGRAPHIC bearing (0=N, 90=E) using the scene's own GCP
         transform. Sentinel-1 GRD images are in radar (slant/ground-range)
         geometry, NOT north-up -- an image-space "up" is the satellite's
         flight direction, which is rotated ~10-15 deg off north and
         flipped between ascending and descending passes. Using image-space
         bearings directly would aim every Tier 2 ray in the wrong direction.
       - Both 180-deg-apart candidates are kept (bearing_deg_a, bearing_deg_b):
         a real-valued FFT is Hermitian-symmetric, so it physically cannot
         tell "this way" from "opposite way". Tier 2 resolves it.

HOW TO RUN
----------
Kaggle/Colab notebook cell (paste or %run): set MODE below to "selftest"
first, run it, then set MODE = "run" and run again.

Terminal:
    python tier1_fft_backtrack.py selftest
    python tier1_fft_backtrack.py run

Output: tier1_fft_results.parquet
"""

import sys
import glob
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

# ======================= EDIT THESE =======================
MODE = "run"   # "selftest" first, then "run" (used when run in a notebook)

DETECTIONS_PARQUET = "/kaggle/input/datasets/mayankgupta005/internal-wave-detection-metadata/real_internal_wave_detections.parquet"
TILES_DIR = "/kaggle/input/datasets/mayankgupta005/sar-image-tiles-june-2021/sar_images_for_iw_detection"          # same folder as IMAGE_DIR in infer_iw_on_real_scenes.py
SAFE_ROOT = "/kaggle/input/datasets/andrejkarpathy/sentinel-1-june-2021-raw-safe"       # same folder as SAFE_ROOT in infer_iw_on_real_scenes.py
OUTPUT_PARQUET = "tier1_fft_results.parquet"
LAND_MASK_PATH = "/kaggle/input/datasets/mayankgupta005/internal-wave-detection-metadata/land_ocean_mask.npz"  # same file infer_iw_on_real_scenes.py uses

# These two are confirmed from convert_scenes_to_images.py
# (TILE_SIZE_KM = 25, OUTPUT_PIXELS = 1024) -- no need to change them
# unless you changed those settings there.
TILE_SIZE_KM = 25
OUTPUT_PIXELS = 1024
# ==========================================================

NATIVE_PX_M = 10.0                                    # S1 IW GRD pixel spacing
NATIVE_TILE_PX = int(TILE_SIZE_KM * 1000 / NATIVE_PX_M)  # 2500
TILE_TO_SCENE = NATIVE_TILE_PX / OUTPUT_PIXELS        # 1 tile px = 2.44 scene px
PIXEL_SCALE_M = TILE_SIZE_KM * 1000 / OUTPUT_PIXELS   # ~24.4 m per tile px

CROP_PAD_PX = 40
MIN_CROP_SIDE_PX = 48

# v2 spectral search settings (see extract_dominant_peak docstring)
LAMBDA_MIN_PX = 250.0 / PIXEL_SCALE_M   # ignore wavelengths < 250 m (~10 px): speckle scale
MIN_PERIODS_IN_CROP = 3.0               # search only wavelengths fitting >=3 times in the crop
HARMONIC_FRAC = 0.35                    # sub-harmonic must hold >=35% of peak's whitened power

# Reliability flags. SNR = peak power / fitted red-noise background.
# SNR_RELIABLE = 20 is the 99th percentile of SNR on wave-free crops (pure
# red noise + range ramp + speckle: median 6.8, 99th pct 20.0, max 22.1).
# On synthetic rank-ordered packets in heavy clutter, crops passing BOTH
# flags had direction and wavelength right 159/159; the rest ~59% / ~51%.
SNR_RELIABLE = 20.0
PERIODS_RELIABLE = 5.0

M_PER_DEG_LAT = 110_540.0
M_PER_DEG_LON_EQ = 111_320.0


# ----------------------------------------------------------------------
# Core FFT routine
# ----------------------------------------------------------------------
def image_bearing_from_freq(fy, fx):
    """Bearing in IMAGE space (0 = image-up, 90 = image-right)."""
    return np.degrees(np.arctan2(fx, -fy)) % 360.0


def extract_dominant_peak(crop, pixel_scale_m):
    """
    v2 spectral peak finder. Returns dict (incl. raw freq_y/freq_x) or None.

    v1 (plain FFT argmax) failed on real data: SAR spectra are 'red' (power
    rises steeply toward low frequency), so the argmax always landed on the
    first bin outside the DC exclusion -- 232/346 real detections peaked at
    exactly 2.2 bins, giving 'wavelengths' of ~half the crop size. v2 fixes:
      1. Remove a best-fit plane (incidence-angle ramp / large brightness trend).
      2. Whiten: divide each bin by the mean power of its frequency annulus,
         so the peak is a DIRECTIONAL excess over the background at that
         scale, not just 'the lowest frequency available'.
      3. Search only wavelengths in [250 m, crop/3] -- longer can't be
         measured from a crop holding <3 periods.
      4. Harmonic check: sharp sech^2 solitons put strong energy at 2x-5x the
         fundamental; if a sub-harmonic (f/n) in the same direction holds
         >= HARMONIC_FRAC of the peak power, take it as the fundamental.
    spectral_peak_snr is the whitened peak power (~1 = background level).
    """
    h, w = crop.shape
    if h < MIN_CROP_SIDE_PX or w < MIN_CROP_SIDE_PX:
        return None

    c = crop.astype(np.float64)
    yy, xx = np.mgrid[0:h, 0:w]
    A = np.c_[np.ones(h * w), xx.ravel(), yy.ravel()]
    coef, *_ = np.linalg.lstsq(A, c.ravel(), rcond=None)
    c = (c - (A @ coef).reshape(h, w)) * np.outer(np.hanning(h), np.hanning(w))

    power = np.abs(np.fft.fftshift(np.fft.fft2(c))) ** 2
    cy, cx = h // 2, w // 2
    fy = (yy - cy) / h
    fx = (xx - cx) / w
    fr = np.hypot(fy, fx)

    with np.errstate(divide="ignore"):
        lam_px = np.where(fr > 0, 1.0 / fr, np.inf)
    band = (lam_px >= LAMBDA_MIN_PX) & (lam_px <= min(h, w) / MIN_PERIODS_IN_CROP)
    if not band.any():
        return None

    # Background spectrum: median power per frequency annulus (a few strong
    # signal bins can't move a median), then a smooth power-law fit across
    # radius, so the background is the red-noise trend, not the signal itself.
    rbin = np.round(fr * min(h, w)).astype(int)
    radii = np.unique(rbin[band])
    medians = np.array([np.median(power[(rbin == r) & band]) for r in radii])
    good = (radii > 0) & (medians > 0)
    if good.sum() >= 3:
        slope, icpt = np.polyfit(np.log(radii[good]), np.log(medians[good]), 1)
        background = np.exp(icpt + slope * np.log(np.maximum(rbin, 1)))
    else:
        background = np.full_like(power, np.median(power[band]))
    whitened = power / np.maximum(background, 1e-30)
    searched = np.where(band, whitened, -np.inf)

    py, px = np.unravel_index(np.argmax(searched), searched.shape)
    peak_snr = whitened[py, px]
    best_y, best_x, best_snr, harmonic_n = py, px, peak_snr, 1
    for n in (2, 3, 4, 5):
        y0 = int(round(cy + (py - cy) / n))
        x0 = int(round(cx + (px - cx) / n))
        ys = slice(max(0, y0 - 1), y0 + 2)
        xs = slice(max(0, x0 - 1), x0 + 2)
        sub = searched[ys, xs]
        if sub.size and np.isfinite(sub).any() and sub.max() >= HARMONIC_FRAC * peak_snr:
            iy, ix = np.unravel_index(np.argmax(sub), sub.shape)
            cand_y, cand_x = ys.start + iy, xs.start + ix
            # A true fundamental of a soliton train carries at least as much
            # RAW power as its harmonics; window leakage below a pure
            # periodicity does not (it only looks big after whitening).
            if power[cand_y, cand_x] >= 0.5 * power[py, px]:
                best_y, best_x = cand_y, cand_x
                best_snr, harmonic_n = whitened[best_y, best_x], n

    freq_y, freq_x = fy[best_y, best_x], fx[best_y, best_x]
    freq_mag = np.hypot(freq_y, freq_x)
    if freq_mag <= 0:
        return None
    wavelength_px = 1.0 / freq_mag

    return {
        "freq_y": float(freq_y),
        "freq_x": float(freq_x),
        "wavelength_m": float(wavelength_px * pixel_scale_m),
        "image_bearing_deg": float(image_bearing_from_freq(freq_y, freq_x)),
        "spectral_peak_snr": float(best_snr),
        "periods_in_crop": float(min(h, w) / wavelength_px),
        "harmonic_n": int(harmonic_n),
    }


# ----------------------------------------------------------------------
# Image-space -> geographic bearing via the scene's GCP transform
# ----------------------------------------------------------------------
def geographic_bearing(transform, cx_scene, cy_scene, freq_y, freq_x, half_len_px=100.0):
    """
    Convert an image-space wavevector to a geographic propagation bearing.

    We transform the CREST direction (perpendicular to the wavevector),
    not the wavevector itself: a crest line is a physical displacement
    and maps correctly through the pixel->lon/lat transform, whereas a
    normal vector does not under a non-conformal mapping (lon/lat degrees
    are not equal-length). The propagation bearing is then the crest
    tangent rotated 90 deg, measured in local east/north METERS.
    Returns (bearing_a, bearing_b) -- 180 deg apart.
    """
    k = np.array([freq_x, freq_y])          # (x=col, y=row)
    k = k / np.linalg.norm(k)
    t = np.array([-k[1], k[0]])             # crest tangent in pixel space

    p1 = (cx_scene - half_len_px * t[0], cy_scene - half_len_px * t[1])
    p2 = (cx_scene + half_len_px * t[0], cy_scene + half_len_px * t[1])
    lon1, lat1 = transform * p1
    lon2, lat2 = transform * p2

    mean_lat = np.radians((lat1 + lat2) / 2)
    east = (lon2 - lon1) * M_PER_DEG_LON_EQ * np.cos(mean_lat)
    north = (lat2 - lat1) * M_PER_DEG_LAT

    crest_bearing = np.degrees(np.arctan2(east, north)) % 360.0
    bearing_a = (crest_bearing + 90.0) % 360.0
    bearing_b = (crest_bearing + 270.0) % 360.0
    return float(bearing_a), float(bearing_b)


# ----------------------------------------------------------------------
# Scene transform loading (same method as infer_iw_on_real_scenes.py)
# ----------------------------------------------------------------------
def find_measurement_tiff(safe_dir, polarization="vv"):
    matches = glob.glob(os.path.join(safe_dir, "measurement", f"*{polarization}*.tiff"))
    if not matches:
        matches = glob.glob(os.path.join(safe_dir, "**", "measurement",
                                         f"*{polarization}*.tiff"), recursive=True)
    if not matches:
        raise FileNotFoundError(f"No {polarization} measurement GeoTIFF under {safe_dir}")
    return matches[0]


def load_scene_transform(scene_id):
    import rasterio
    from rasterio.transform import from_gcps
    tiff = find_measurement_tiff(os.path.join(SAFE_ROOT, f"{scene_id}.SAFE"))
    with rasterio.open(tiff) as src:
        gcps, _ = src.get_gcps()
        if not gcps:
            raise ValueError(f"No GCPs in {tiff}")
        return from_gcps(gcps)


# ----------------------------------------------------------------------
# Tile-filename lookup + land mask (for parquets from older inference runs)
# ----------------------------------------------------------------------
TILE_FILENAME_RE = re.compile(r"^(.+)_tile(\d+)_row(\d+)_col(\d+)\.png$")


def build_tile_lookup(tiles_dir):
    """Map (scene_id, tile_idx) -> tile filename by scanning TILES_DIR.
    Older parquets only store scene_id + tile_idx, not the filename."""
    lookup = {}
    for p in Path(tiles_dir).glob("*.png"):
        m = TILE_FILENAME_RE.match(p.name)
        if m:
            lookup[(m.group(1), int(m.group(2)))] = p.name
    return lookup


def load_land_checker(path):
    """Same lookup as infer_iw_on_real_scenes.py's is_on_land."""
    d = np.load(path)
    mask, mlat, mlon = d["mask"], d["lat"], d["lon"]

    def is_on_land(lat, lon):
        lat = max(mlat.min(), min(mlat.max(), lat))
        lon = max(mlon.min(), min(mlon.max(), lon))
        i = int((lat - mlat[0]) / (mlat[1] - mlat[0]))
        j = int((lon - mlon[0]) / (mlon[1] - mlon[0]))
        return not bool(mask[i, j])   # mask True = ocean
    return is_on_land


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------
def get_box_xyxy(row):
    """Read the detection box as [x1, y1, x2, y2] in 1024-tile pixels."""
    if "box_resized" in row.index and row["box_resized"] is not None:
        b = np.asarray(row["box_resized"], dtype=float)
        return b[0], b[1], b[2], b[3]
    if {"bbox_x", "bbox_y", "bbox_w", "bbox_h"}.issubset(row.index):
        return (row["bbox_x"], row["bbox_y"],
                row["bbox_x"] + row["bbox_w"], row["bbox_y"] + row["bbox_h"])
    raise KeyError("No box column found (expected 'box_resized')")


def crop_box(tile, x1, y1, x2, y2, pad):
    h, w = tile.shape
    return tile[max(0, int(y1) - pad):min(h, int(y2) + pad),
                max(0, int(x1) - pad):min(w, int(x2) + pad)]


def angular_diff(a, b):
    d = abs(a - b) % 360.0
    return min(d, 360.0 - d)


def make_synthetic_stripe(h, w, wavelength_px, image_bearing_deg, noise_std=0.05, seed=0):
    rng = np.random.default_rng(seed)
    fx = np.sin(np.radians(image_bearing_deg))
    fy = -np.cos(np.radians(image_bearing_deg))
    yy, xx = np.mgrid[0:h, 0:w]
    img = np.sin(2 * np.pi * (fx * xx + fy * yy) / wavelength_px)
    return img + rng.normal(0, noise_std, size=img.shape)


def _red_noise(h, w, rng, beta=2.0):
    fy = np.fft.fftfreq(h)[:, None]
    fx = np.fft.fftfreq(w)[None, :]
    f = np.hypot(fy, fx)
    f[0, 0] = 1
    spec = (rng.normal(size=(h, w)) + 1j * rng.normal(size=(h, w))) / f ** (beta / 2)
    n = np.real(np.fft.ifft2(spec))
    return (n - n.mean()) / n.std()


def make_synthetic_packet(h, w, spacing_px, image_bearing_deg, rng):
    """Rank-ordered sech^2 soliton packet occupying part of the crop, on a
    red-noise background with a range ramp, a dark slick and 4-look speckle."""
    yy, xx = np.mgrid[0:h, 0:w]
    ex, ny = np.sin(np.radians(image_bearing_deg)), np.cos(np.radians(image_bearing_deg))
    s = ex * (xx - w / 2) - ny * (yy - h / 2)          # along propagation
    t = ny * (xx - w / 2) + ex * (yy - h / 2)          # along crest
    L = 0.8 * min(h, w)
    sig = np.zeros((h, w))
    for i in range(max(3, int(L / spacing_px))):
        sig += (1.0 - 0.12 * i) / np.cosh((s - (L / 2 - i * spacing_px)) / (spacing_px / 5)) ** 2
    envelope = np.exp(-(t / (0.45 * max(h, w))) ** 4) * (np.abs(s) < L / 2 + spacing_px)
    slick = -0.8 * np.exp(-(((xx - 0.3 * w) / (0.35 * w)) ** 2 + ((yy - 0.7 * h) / (0.3 * h)) ** 2))
    img = 1.0 + 0.7 * sig * envelope + 0.75 * _red_noise(h, w, rng) + 1.2 * xx / w + slick
    return np.clip(img, 0.05, None) * rng.gamma(4, 0.25, size=(h, w))


# ----------------------------------------------------------------------
# Self-test
# ----------------------------------------------------------------------
def selftest():
    from affine import Affine
    print("=" * 70)
    print("TIER 1 SELF-TEST")
    print("=" * 70)
    ok_all = True

    print("\n[1] Wavelength + image-space bearing on synthetic stripes")
    for wl, brg in [(30, 0), (30, 90), (45, 135), (60, 200), (20, 315)]:
        r = extract_dominant_peak(make_synthetic_stripe(256, 256, wl, brg), 1.0)
        b_a = r["image_bearing_deg"]
        ok = (abs(r["wavelength_m"] - wl) < 3.5
              and min(angular_diff(b_a, brg), angular_diff(b_a + 180, brg)) < 10)
        ok_all &= ok
        print(f"  {'PASS' if ok else 'FAIL'}  true {wl}px @ {brg}deg -> "
              f"{r['wavelength_m']:.1f}px @ {b_a:.1f}/{(b_a+180)%360:.1f}deg")

    print("\n[2] Image -> geographic bearing through a rotated GCP-style transform")
    # Build pixel->lon/lat transforms like a real S1 scene: 10 m pixels,
    # image rotated by `rot` deg relative to north, at lat ~12N.
    lat0, lon0 = 12.0, 90.0
    dlat = NATIVE_PX_M / M_PER_DEG_LAT
    dlon = NATIVE_PX_M / (M_PER_DEG_LON_EQ * np.cos(np.radians(lat0)))
    for rot in [0.0, -12.0, 15.0, 168.0]:   # ~ascending, descending-like headings
        th = np.radians(rot)
        # pixel (x right, y down) -> (east, north) meters, rotated clockwise by rot:
        #   east  =  cos(th)*x - sin(th)*y
        #   north = -sin(th)*x - cos(th)*y
        # (a vector at image bearing b then lands at geographic bearing b+rot)
        a_e, b_e = np.cos(th) * NATIVE_PX_M, -np.sin(th) * NATIVE_PX_M
        d_n, e_n = -np.sin(th) * NATIVE_PX_M, -np.cos(th) * NATIVE_PX_M
        T = Affine(a_e / NATIVE_PX_M * dlon, b_e / NATIVE_PX_M * dlon, lon0,
                   d_n / NATIVE_PX_M * dlat, e_n / NATIVE_PX_M * dlat, lat0)
        for img_brg in [0.0, 70.0, 225.0]:
            r = extract_dominant_peak(make_synthetic_stripe(256, 256, 40, img_brg), 1.0)
            ga, gb = geographic_bearing(T, 1000, 1000, r["freq_y"], r["freq_x"])
            expected = (img_brg + rot) % 360.0
            ok = min(angular_diff(ga, expected), angular_diff(gb, expected)) < 10
            ok_all &= ok
            print(f"  {'PASS' if ok else 'FAIL'}  image rot {rot:+6.1f}deg, image bearing "
                  f"{img_brg:5.1f} -> geo {ga:5.1f}/{gb:5.1f} (expected {expected:5.1f})")

    print("\n[3] Realistic packets: rank-ordered solitons inside a cluttered crop")
    print("    (red-noise background, range ramp, dark slick, speckle -- the setting")
    print("     where v1 locked onto crop-scale trends on real data)")
    rng = np.random.default_rng(11)
    n = n_rel = rel_ok = stuck = 0
    for sp_m in [600, 900, 1200, 1500, 2000]:
        for brg in np.linspace(0, 330, 12):
            for (h, w) in [(700, 500), (560, 480), (900, 400)]:
                img = make_synthetic_packet(h, w, sp_m / PIXEL_SCALE_M, brg, rng)
                r = extract_dominant_peak(img, PIXEL_SCALE_M)
                n += 1
                stuck += r["wavelength_m"] > 0.3 * min(h, w) * PIXEL_SCALE_M
                if r["spectral_peak_snr"] >= SNR_RELIABLE and r["periods_in_crop"] >= PERIODS_RELIABLE:
                    n_rel += 1
                    d = abs(r["image_bearing_deg"] - brg) % 180
                    rel_ok += (min(d, 180 - d) < 15
                               and abs(r["wavelength_m"] - sp_m) / sp_m < 0.25)
    acc = rel_ok / max(n_rel, 1)
    ok3 = acc >= 0.8 and stuck / n < 0.1 and n_rel >= 0.25 * n
    ok_all &= ok3
    print(f"  {'PASS' if ok3 else 'FAIL'}  peaks stuck at crop scale: {stuck}/{n} "
          f"| flagged reliable: {n_rel}/{n} | reliable & correct: {rel_ok}/{n_rel} ({acc:.0%})")

    print("\n" + "=" * 70)
    print("ALL SELF-TESTS PASSED." if ok_all else "SELF-TEST FAILURES -- do not run on real data.")
    print("=" * 70)
    return ok_all


# ----------------------------------------------------------------------
# Real run
# ----------------------------------------------------------------------
def run():
    det_path, tiles_dir = Path(DETECTIONS_PARQUET), Path(TILES_DIR)
    for p, what in [(det_path, "DETECTIONS_PARQUET"), (tiles_dir, "TILES_DIR"),
                    (Path(SAFE_ROOT), "SAFE_ROOT")]:
        if not p.exists():
            print(f"ERROR: {what} not found: {p}")
            return

    det = pd.read_parquet(det_path)
    print(f"Loaded {len(det)} detections. Columns: {list(det.columns)}")
    print(f"Tile scale: {PIXEL_SCALE_M:.2f} m/px "
          f"({TILE_SIZE_KM} km tiles resampled to {OUTPUT_PIXELS}px)")

    # Land mask: older parquets were written before land-masking existed
    # (the original 798-detection run had ~450 detections on land). Always
    # re-apply it here so Tier 1 never measures rivers/deltas.
    if not Path(LAND_MASK_PATH).exists():
        print(f"ERROR: LAND_MASK_PATH not found: {LAND_MASK_PATH}")
        return
    is_on_land = load_land_checker(LAND_MASK_PATH)
    on_land = det.apply(lambda r: is_on_land(float(r["real_lat"]), float(r["real_lon"])), axis=1)
    print(f"Land mask: dropping {int(on_land.sum())} of {len(det)} detections on land")
    det = det[~on_land].reset_index(drop=True)

    # Tile filename: present in newer parquets, rebuilt from scene_id + tile_idx otherwise
    if "tile_filename" not in det.columns:
        lookup = build_tile_lookup(tiles_dir)
        print(f"No 'tile_filename' column -- matched via scene_id + tile_idx "
              f"against {len(lookup)} tiles in TILES_DIR")
        det["tile_filename"] = [lookup.get((s, int(t))) for s, t in
                                zip(det["scene_id"], det["tile_idx"])]

    tile_cache, transform_cache = {}, {}
    results = []
    n_missing_tile = n_bad_crop = n_no_transform = n_low = 0

    for i, (_, row) in enumerate(det.iterrows()):
        fname = row["tile_filename"]
        if fname is None:
            n_missing_tile += 1
            continue
        if fname not in tile_cache:
            tp = tiles_dir / fname
            tile_cache[fname] = np.array(Image.open(tp).convert("L")) if tp.exists() else None
        tile = tile_cache[fname]
        if tile is None:
            n_missing_tile += 1
            continue

        x1, y1, x2, y2 = get_box_xyxy(row)
        crop = crop_box(tile, x1, y1, x2, y2, CROP_PAD_PX)
        peak = extract_dominant_peak(crop, PIXEL_SCALE_M)
        if peak is None:
            n_bad_crop += 1
            continue

        scene_id = row["scene_id"]
        if scene_id not in transform_cache:
            try:
                transform_cache[scene_id] = load_scene_transform(scene_id)
            except Exception as e:
                print(f"  WARNING: no GCP transform for {scene_id}: {e}")
                transform_cache[scene_id] = None
        T = transform_cache[scene_id]

        if T is not None:
            geo_a, geo_b = geographic_bearing(
                T, row["scene_pixel_x"], row["scene_pixel_y"],
                peak["freq_y"], peak["freq_x"],
            )
        else:
            geo_a = geo_b = np.nan
            n_no_transform += 1

        low = (not np.isfinite(peak["spectral_peak_snr"])
               or peak["spectral_peak_snr"] < SNR_RELIABLE
               or peak["periods_in_crop"] < PERIODS_RELIABLE
               or T is None)
        n_low += int(low)

        results.append({
            "scene_id": scene_id,
            "tile_filename": fname,
            "real_lat": row.get("real_lat"),
            "real_lon": row.get("real_lon"),
            "confidence": row.get("confidence"),
            "crop_h": crop.shape[0], "crop_w": crop.shape[1],
            "wavelength_m": peak["wavelength_m"],
            "image_bearing_deg": peak["image_bearing_deg"],
            "bearing_deg_a": geo_a,
            "bearing_deg_b": geo_b,
            "spectral_peak_snr": peak["spectral_peak_snr"],
            "periods_in_crop": peak["periods_in_crop"],
            "harmonic_n": peak["harmonic_n"],
            "low_confidence": low,
        })
        if (i + 1) % 100 == 0:
            print(f"  processed {i + 1}/{len(det)}")

    out = pd.DataFrame(results)
    out.to_parquet(OUTPUT_PARQUET, index=False)

    print(f"\n{'=' * 60}\nTIER 1 SUMMARY\n{'=' * 60}")
    print(f"Detections in:              {len(det)}")
    print(f"Processed:                  {len(out)}")
    print(f"Tile file not found:        {n_missing_tile}")
    print(f"Crop too small/degenerate:  {n_bad_crop}")
    print(f"No GCP transform (bearing NaN): {n_no_transform}")
    print(f"Low-confidence flagged:     {n_low} ({100 * n_low / max(1, len(out)):.1f}%)"
          f"  [SNR < {SNR_RELIABLE} or < {PERIODS_RELIABLE} periods in crop]")
    if len(out):
        rel = out[~out["low_confidence"]]
        print(f"Reliable (go to Tier 2):    {len(rel)}")
        print("\nWavelength (m), reliable only:")
        print(rel["wavelength_m"].describe().round(1).to_string())
        print("\nWavelength (m), all:")
        print(out["wavelength_m"].describe().round(1).to_string())
        print("\nWhitened spectral SNR (all):")
        print(out["spectral_peak_snr"].describe().round(1).to_string())
        print("\nPeriods in crop (all):")
        print(out["periods_in_crop"].describe().round(1).to_string())
        print(f"\nHarmonic correction applied: {(out['harmonic_n'] > 1).sum()}")
    print(f"\nSaved to {OUTPUT_PARQUET}")


def main():
    # In a notebook, sys.argv holds Jupyter's own kernel args -- ignore them
    # and use MODE. From a terminal, a 'selftest'/'run' argument wins.
    in_notebook = "ipykernel" in sys.modules
    mode = MODE
    if not in_notebook and len(sys.argv) == 2 and sys.argv[1] in ("selftest", "run"):
        mode = sys.argv[1]
    print(f"Mode: {mode}")
    if mode == "selftest":
        selftest()
    elif mode == "run":
        run()
    else:
        print(f"Unknown MODE {mode!r}; use 'selftest' or 'run'.")


if __name__ == "__main__":
    main()

Mode: run
Loaded 798 detections. Columns: ['scene_id', 'tile_idx', 'confidence', 'box_resized', 'scene_pixel_x', 'scene_pixel_y', 'real_lon', 'real_lat']
Tile scale: 24.41 m/px (25 km tiles resampled to 1024px)
Land mask: dropping 452 of 798 detections on land
No 'tile_filename' column -- matched via scene_id + tile_idx against 4524 tiles in TILES_DIR
  processed 100/346
  processed 200/346
  processed 300/346

TIER 1 SUMMARY
Detections in:              346
Processed:                  346
Tile file not found:        0
Crop too small/degenerate:  0
No GCP transform (bearing NaN): 0
Low-confidence flagged:     241 (69.7%)  [SNR < 20.0 or < 5.0 periods in crop]
Reliable (go to Tier 2):    105

Wavelength (m), reliable only:
count     105.0
mean     1139.2
std      1018.4
min       250.0
25%       312.3
50%       861.5
75%      1727.0
max      4219.0

Wavelength (m), all:
count     346.0
mean     2459.0
std      1473.2
min       250.0
25%      1254.8
50%      2471.8
75%      3429.8
max    

In [6]:
""" V3
OceanXRay — Tier 1: FFT-based wavelength & propagation-direction extraction
=============================================================================
For each land-filtered internal-wave detection in
real_internal_wave_detections.parquet (written by infer_iw_on_real_scenes.py):

  1. Load the source SAR tile the detection came from.
  2. Crop a padded window around the detection box (`box_resized`,
     [x1, y1, x2, y2] in the 1024x1024 tile's pixel space).
  3. Remove the large-scale brightness plane, apply a 2D Hann window.
  4. 2D FFT, divide by a fitted red-noise background, and take the strongest
     directional peak among wavelengths 250 m .. crop/3 (with a harmonic
     check). v1 skipped the background step and on real data latched onto
     crop-scale brightness trends instead of soliton spacing -- see
     extract_dominant_peak.
     Detections are flagged low_confidence unless SNR >= 20 (above the
     99th percentile of wave-free crops) and >= 5 periods fit in the crop.
  5. Convert the peak to:
       - wavelength in meters (tile pixel scale ~24.4 m/px)
       - propagation direction, first in IMAGE space, then converted to a
         real GEOGRAPHIC bearing (0=N, 90=E) using the scene's own GCP
         transform. Sentinel-1 GRD images are in radar (slant/ground-range)
         geometry, NOT north-up -- an image-space "up" is the satellite's
         flight direction, which is rotated ~10-15 deg off north and
         flipped between ascending and descending passes. Using image-space
         bearings directly would aim every Tier 2 ray in the wrong direction.
       - Both 180-deg-apart candidates are kept (bearing_deg_a, bearing_deg_b):
         a real-valued FFT is Hermitian-symmetric, so it physically cannot
         tell "this way" from "opposite way". Tier 2 resolves it.

HOW TO RUN
----------
Kaggle/Colab notebook cell (paste or %run): set MODE below to "selftest"
first, run it, then MODE = "run", then MODE = "inspect" (saves
tier1_inspect.png: the top reliable crops with the estimated axis drawn).

Terminal:
    python tier1_fft_backtrack.py selftest
    python tier1_fft_backtrack.py run

Output: tier1_fft_results.parquet
"""

import sys
import glob
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

# ======================= EDIT THESE =======================
MODE = "inspect"   # "selftest" -> "run" -> "inspect" (used when run in a notebook)

DETECTIONS_PARQUET = "/kaggle/input/datasets/mayankgupta005/internal-wave-detection-metadata/real_internal_wave_detections.parquet"
TILES_DIR = "/kaggle/input/datasets/mayankgupta005/sar-image-tiles-june-2021/sar_images_for_iw_detection"          # same folder as IMAGE_DIR in infer_iw_on_real_scenes.py
SAFE_ROOT = "/kaggle/input/datasets/andrejkarpathy/sentinel-1-june-2021-raw-safe"       # same folder as SAFE_ROOT in infer_iw_on_real_scenes.py
OUTPUT_PARQUET = "tier1_fft_results.parquet"
LAND_MASK_PATH = "/kaggle/input/datasets/mayankgupta005/internal-wave-detection-metadata/land_ocean_mask.npz"  # same file infer_iw_on_real_scenes.py uses

# These two are confirmed from convert_scenes_to_images.py
# (TILE_SIZE_KM = 25, OUTPUT_PIXELS = 1024) -- no need to change them
# unless you changed those settings there.
TILE_SIZE_KM = 25
OUTPUT_PIXELS = 1024
# ==========================================================

NATIVE_PX_M = 10.0                                    # S1 IW GRD pixel spacing
NATIVE_TILE_PX = int(TILE_SIZE_KM * 1000 / NATIVE_PX_M)  # 2500
TILE_TO_SCENE = NATIVE_TILE_PX / OUTPUT_PIXELS        # 1 tile px = 2.44 scene px
PIXEL_SCALE_M = TILE_SIZE_KM * 1000 / OUTPUT_PIXELS   # ~24.4 m per tile px

CROP_PAD_PX = 40
MIN_CROP_SIDE_PX = 48

# v2 spectral search settings (see extract_dominant_peak docstring)
# Search floor 500 m: ocean swell (8-17 s periods -> ~100-450 m wavelength,
# strongest during the June SW monsoon) is the main periodic confounder in
# SAR. The v2 run on real data put 49/105 'reliable' results at 250-500 m
# with a median ~34 periods per crop -- a crop-filling texture, i.e. swell,
# not a soliton packet (IW Atlas, Andaman Sea: soliton surface width
# 600-1000 m). Swell energy is still measured and reported separately.
LAMBDA_MIN_PX = 500.0 / PIXEL_SCALE_M
SWELL_BAND_M = (100.0, 500.0)
MIN_PERIODS_IN_CROP = 3.0               # search only wavelengths fitting >=3 times in the crop
HARMONIC_FRAC = 0.35                    # sub-harmonic must hold >=35% of peak's whitened power

# Reliability flags. SNR = peak power / fitted red-noise background.
# SNR_RELIABLE = 20 is the 99th percentile of SNR on wave-free crops (pure
# red noise + range ramp + speckle: median 6.8, 99th pct 20.0, max 22.1).
# On synthetic rank-ordered packets in heavy clutter, crops passing BOTH
# flags had direction and wavelength right 159/159; the rest ~59% / ~51%.
SNR_RELIABLE = 20.0
PERIODS_RELIABLE = 5.0

M_PER_DEG_LAT = 110_540.0
M_PER_DEG_LON_EQ = 111_320.0


# ----------------------------------------------------------------------
# Core FFT routine
# ----------------------------------------------------------------------
def image_bearing_from_freq(fy, fx):
    """Bearing in IMAGE space (0 = image-up, 90 = image-right)."""
    return np.degrees(np.arctan2(fx, -fy)) % 360.0


def extract_dominant_peak(crop, pixel_scale_m):
    """
    v2 spectral peak finder. Returns dict (incl. raw freq_y/freq_x) or None.

    v1 (plain FFT argmax) failed on real data: SAR spectra are 'red' (power
    rises steeply toward low frequency), so the argmax always landed on the
    first bin outside the DC exclusion -- 232/346 real detections peaked at
    exactly 2.2 bins, giving 'wavelengths' of ~half the crop size. v2 fixes:
      1. Remove a best-fit plane (incidence-angle ramp / large brightness trend).
      2. Whiten: divide each bin by the mean power of its frequency annulus,
         so the peak is a DIRECTIONAL excess over the background at that
         scale, not just 'the lowest frequency available'.
      3. Search only wavelengths in [250 m, crop/3] -- longer can't be
         measured from a crop holding <3 periods.
      4. Harmonic check: sharp sech^2 solitons put strong energy at 2x-5x the
         fundamental; if a sub-harmonic (f/n) in the same direction holds
         >= HARMONIC_FRAC of the peak power, take it as the fundamental.
    spectral_peak_snr is the whitened peak power (~1 = background level).
    """
    h, w = crop.shape
    if h < MIN_CROP_SIDE_PX or w < MIN_CROP_SIDE_PX:
        return None

    c = crop.astype(np.float64)
    yy, xx = np.mgrid[0:h, 0:w]
    A = np.c_[np.ones(h * w), xx.ravel(), yy.ravel()]
    coef, *_ = np.linalg.lstsq(A, c.ravel(), rcond=None)
    c = (c - (A @ coef).reshape(h, w)) * np.outer(np.hanning(h), np.hanning(w))

    power = np.abs(np.fft.fftshift(np.fft.fft2(c))) ** 2
    cy, cx = h // 2, w // 2
    fy = (yy - cy) / h
    fx = (xx - cx) / w
    fr = np.hypot(fy, fx)

    with np.errstate(divide="ignore"):
        lam_px = np.where(fr > 0, 1.0 / fr, np.inf)
    band = (lam_px >= LAMBDA_MIN_PX) & (lam_px <= min(h, w) / MIN_PERIODS_IN_CROP)
    if not band.any():
        return None

    # Background spectrum: median power per frequency annulus (a few strong
    # signal bins can't move a median), then a smooth power-law fit across
    # radius, so the background is the red-noise trend, not the signal itself.
    rbin = np.round(fr * min(h, w)).astype(int)
    radii = np.unique(rbin[band])
    medians = np.array([np.median(power[(rbin == r) & band]) for r in radii])
    good = (radii > 0) & (medians > 0)
    if good.sum() >= 3:
        slope, icpt = np.polyfit(np.log(radii[good]), np.log(medians[good]), 1)
        background = np.exp(icpt + slope * np.log(np.maximum(rbin, 1)))
    else:
        background = np.full_like(power, np.median(power[band]))
    whitened = power / np.maximum(background, 1e-30)
    searched = np.where(band, whitened, -np.inf)

    py, px = np.unravel_index(np.argmax(searched), searched.shape)
    peak_snr = whitened[py, px]
    best_y, best_x, best_snr, harmonic_n = py, px, peak_snr, 1
    # Harmonic check, n = 2, 3 only. Accept a sub-harmonic only if it is a
    # genuine spectral peak on its own: a strict local maximum of the
    # whitened spectrum AND above the reliability SNR. (An earlier
    # 'raw power >= 50%' rule fired on 61% of real crops and dragged
    # estimates to the long-wavelength limit, because real SAR spectra
    # always carry more raw power at lower frequencies.)
    from scipy.ndimage import maximum_filter
    local_max = whitened >= maximum_filter(whitened, size=5)
    for n in (2, 3):
        y0 = int(round(cy + (py - cy) / n))
        x0 = int(round(cx + (px - cx) / n))
        ys = slice(max(0, y0 - 1), y0 + 2)
        xs = slice(max(0, x0 - 1), x0 + 2)
        sub = np.where(local_max[ys, xs], searched[ys, xs], -np.inf)
        if not (sub.size and np.isfinite(sub).any()):
            continue
        if sub.max() >= max(HARMONIC_FRAC * peak_snr, SNR_RELIABLE):
            iy, ix = np.unravel_index(np.argmax(sub), sub.shape)
            best_y, best_x = ys.start + iy, xs.start + ix
            best_snr, harmonic_n = whitened[best_y, best_x], n

    # Swell diagnostic: strongest peak in the 100-500 m band, measured against
    # that band's OWN background fit (speckle flattens the spectrum at these
    # scales, so extrapolating the main-band fit would inflate every bin).
    swell_band = ((lam_px >= SWELL_BAND_M[0] / pixel_scale_m)
                  & (lam_px < SWELL_BAND_M[1] / pixel_scale_m))
    swell_snr, swell_wavelength_m = np.nan, np.nan
    s_radii = np.unique(rbin[swell_band])
    if s_radii.size >= 3:
        s_med = np.array([np.median(power[(rbin == r) & swell_band]) for r in s_radii])
        ok = (s_radii > 0) & (s_med > 0)
        if ok.sum() >= 3:
            s_slope, s_icpt = np.polyfit(np.log(s_radii[ok]), np.log(s_med[ok]), 1)
            s_bg = np.exp(s_icpt + s_slope * np.log(np.maximum(rbin, 1)))
            s_white = np.where(swell_band, power / np.maximum(s_bg, 1e-30), -np.inf)
            sy, sx = np.unravel_index(np.argmax(s_white), s_white.shape)
            swell_snr = float(s_white[sy, sx])
            swell_wavelength_m = float(pixel_scale_m / max(fr[sy, sx], 1e-12))

    freq_y, freq_x = fy[best_y, best_x], fx[best_y, best_x]
    freq_mag = np.hypot(freq_y, freq_x)
    if freq_mag <= 0:
        return None
    wavelength_px = 1.0 / freq_mag

    return {
        "freq_y": float(freq_y),
        "freq_x": float(freq_x),
        "wavelength_m": float(wavelength_px * pixel_scale_m),
        "image_bearing_deg": float(image_bearing_from_freq(freq_y, freq_x)),
        "spectral_peak_snr": float(best_snr),
        "periods_in_crop": float(min(h, w) / wavelength_px),
        "harmonic_n": int(harmonic_n),
        "swell_snr": swell_snr,
        "swell_wavelength_m": swell_wavelength_m,
    }


# ----------------------------------------------------------------------
# Image-space -> geographic bearing via the scene's GCP transform
# ----------------------------------------------------------------------
def geographic_bearing(transform, cx_scene, cy_scene, freq_y, freq_x, half_len_px=100.0):
    """
    Convert an image-space wavevector to a geographic propagation bearing.

    We transform the CREST direction (perpendicular to the wavevector),
    not the wavevector itself: a crest line is a physical displacement
    and maps correctly through the pixel->lon/lat transform, whereas a
    normal vector does not under a non-conformal mapping (lon/lat degrees
    are not equal-length). The propagation bearing is then the crest
    tangent rotated 90 deg, measured in local east/north METERS.
    Returns (bearing_a, bearing_b) -- 180 deg apart.
    """
    k = np.array([freq_x, freq_y])          # (x=col, y=row)
    k = k / np.linalg.norm(k)
    t = np.array([-k[1], k[0]])             # crest tangent in pixel space

    p1 = (cx_scene - half_len_px * t[0], cy_scene - half_len_px * t[1])
    p2 = (cx_scene + half_len_px * t[0], cy_scene + half_len_px * t[1])
    lon1, lat1 = transform * p1
    lon2, lat2 = transform * p2

    mean_lat = np.radians((lat1 + lat2) / 2)
    east = (lon2 - lon1) * M_PER_DEG_LON_EQ * np.cos(mean_lat)
    north = (lat2 - lat1) * M_PER_DEG_LAT

    crest_bearing = np.degrees(np.arctan2(east, north)) % 360.0
    bearing_a = (crest_bearing + 90.0) % 360.0
    bearing_b = (crest_bearing + 270.0) % 360.0
    return float(bearing_a), float(bearing_b)


# ----------------------------------------------------------------------
# Scene transform loading (same method as infer_iw_on_real_scenes.py)
# ----------------------------------------------------------------------
def find_measurement_tiff(safe_dir, polarization="vv"):
    matches = glob.glob(os.path.join(safe_dir, "measurement", f"*{polarization}*.tiff"))
    if not matches:
        matches = glob.glob(os.path.join(safe_dir, "**", "measurement",
                                         f"*{polarization}*.tiff"), recursive=True)
    if not matches:
        raise FileNotFoundError(f"No {polarization} measurement GeoTIFF under {safe_dir}")
    return matches[0]


def load_scene_transform(scene_id):
    import rasterio
    from rasterio.transform import from_gcps
    tiff = find_measurement_tiff(os.path.join(SAFE_ROOT, f"{scene_id}.SAFE"))
    with rasterio.open(tiff) as src:
        gcps, _ = src.get_gcps()
        if not gcps:
            raise ValueError(f"No GCPs in {tiff}")
        return from_gcps(gcps)


# ----------------------------------------------------------------------
# Tile-filename lookup + land mask (for parquets from older inference runs)
# ----------------------------------------------------------------------
TILE_FILENAME_RE = re.compile(r"^(.+)_tile(\d+)_row(\d+)_col(\d+)\.png$")


def build_tile_lookup(tiles_dir):
    """Map (scene_id, tile_idx) -> tile filename by scanning TILES_DIR.
    Older parquets only store scene_id + tile_idx, not the filename."""
    lookup = {}
    for p in Path(tiles_dir).glob("*.png"):
        m = TILE_FILENAME_RE.match(p.name)
        if m:
            lookup[(m.group(1), int(m.group(2)))] = p.name
    return lookup


def load_land_checker(path):
    """Same lookup as infer_iw_on_real_scenes.py's is_on_land."""
    d = np.load(path)
    mask, mlat, mlon = d["mask"], d["lat"], d["lon"]

    def is_on_land(lat, lon):
        lat = max(mlat.min(), min(mlat.max(), lat))
        lon = max(mlon.min(), min(mlon.max(), lon))
        i = int((lat - mlat[0]) / (mlat[1] - mlat[0]))
        j = int((lon - mlon[0]) / (mlon[1] - mlon[0]))
        return not bool(mask[i, j])   # mask True = ocean
    return is_on_land


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------
def get_box_xyxy(row):
    """Read the detection box as [x1, y1, x2, y2] in 1024-tile pixels."""
    if "box_resized" in row.index and row["box_resized"] is not None:
        b = np.asarray(row["box_resized"], dtype=float)
        return b[0], b[1], b[2], b[3]
    if {"bbox_x", "bbox_y", "bbox_w", "bbox_h"}.issubset(row.index):
        return (row["bbox_x"], row["bbox_y"],
                row["bbox_x"] + row["bbox_w"], row["bbox_y"] + row["bbox_h"])
    raise KeyError("No box column found (expected 'box_resized')")


def crop_box(tile, x1, y1, x2, y2, pad):
    h, w = tile.shape
    return tile[max(0, int(y1) - pad):min(h, int(y2) + pad),
                max(0, int(x1) - pad):min(w, int(x2) + pad)]


def angular_diff(a, b):
    d = abs(a - b) % 360.0
    return min(d, 360.0 - d)


def make_synthetic_stripe(h, w, wavelength_px, image_bearing_deg, noise_std=0.05, seed=0):
    rng = np.random.default_rng(seed)
    fx = np.sin(np.radians(image_bearing_deg))
    fy = -np.cos(np.radians(image_bearing_deg))
    yy, xx = np.mgrid[0:h, 0:w]
    img = np.sin(2 * np.pi * (fx * xx + fy * yy) / wavelength_px)
    return img + rng.normal(0, noise_std, size=img.shape)


def _red_noise(h, w, rng, beta=2.0):
    fy = np.fft.fftfreq(h)[:, None]
    fx = np.fft.fftfreq(w)[None, :]
    f = np.hypot(fy, fx)
    f[0, 0] = 1
    spec = (rng.normal(size=(h, w)) + 1j * rng.normal(size=(h, w))) / f ** (beta / 2)
    n = np.real(np.fft.ifft2(spec))
    return (n - n.mean()) / n.std()


def make_synthetic_packet(h, w, spacing_px, image_bearing_deg, rng,
                          red_amp=0.75, red_beta=2.0, swell=None):
    """Rank-ordered sech^2 soliton packet occupying part of the crop, on a
    red-noise background with a range ramp, a dark slick and 4-look speckle.
    swell=(amplitude, wavelength_px, image_bearing_deg) adds a crop-filling
    swell field."""
    yy, xx = np.mgrid[0:h, 0:w]
    ex, ny = np.sin(np.radians(image_bearing_deg)), np.cos(np.radians(image_bearing_deg))
    s = ex * (xx - w / 2) - ny * (yy - h / 2)          # along propagation
    t = ny * (xx - w / 2) + ex * (yy - h / 2)          # along crest
    L = 0.8 * min(h, w)
    sig = np.zeros((h, w))
    for i in range(max(3, int(L / spacing_px))):
        sig += (1.0 - 0.12 * i) / np.cosh((s - (L / 2 - i * spacing_px)) / (spacing_px / 5)) ** 2
    envelope = np.exp(-(t / (0.45 * max(h, w))) ** 4) * (np.abs(s) < L / 2 + spacing_px)
    slick = -0.8 * np.exp(-(((xx - 0.3 * w) / (0.35 * w)) ** 2 + ((yy - 0.7 * h) / (0.3 * h)) ** 2))
    img = (1.0 + 0.7 * sig * envelope + red_amp * _red_noise(h, w, rng, red_beta)
           + 1.2 * xx / w + slick)
    if swell is not None:
        s_amp, s_wl, s_brg = swell
        s_e, s_n = np.sin(np.radians(s_brg)), np.cos(np.radians(s_brg))
        img += s_amp * np.sin(2 * np.pi * (s_e * xx - s_n * yy) / s_wl)
    return np.clip(img, 0.05, None) * rng.gamma(4, 0.25, size=(h, w))


# ----------------------------------------------------------------------
# Self-test
# ----------------------------------------------------------------------
def selftest():
    from affine import Affine
    print("=" * 70)
    print("TIER 1 SELF-TEST")
    print("=" * 70)
    ok_all = True

    print("\n[1] Wavelength + image-space bearing on synthetic stripes")
    # (all >= the 500 m / ~20.5 px search floor)
    for wl, brg in [(30, 0), (30, 90), (45, 135), (60, 200), (25, 315)]:
        r = extract_dominant_peak(make_synthetic_stripe(256, 256, wl, brg), 1.0)
        b_a = r["image_bearing_deg"]
        ok = (abs(r["wavelength_m"] - wl) < 3.5
              and min(angular_diff(b_a, brg), angular_diff(b_a + 180, brg)) < 10)
        ok_all &= ok
        print(f"  {'PASS' if ok else 'FAIL'}  true {wl}px @ {brg}deg -> "
              f"{r['wavelength_m']:.1f}px @ {b_a:.1f}/{(b_a+180)%360:.1f}deg")

    print("\n[2] Image -> geographic bearing through a rotated GCP-style transform")
    # Build pixel->lon/lat transforms like a real S1 scene: 10 m pixels,
    # image rotated by `rot` deg relative to north, at lat ~12N.
    lat0, lon0 = 12.0, 90.0
    dlat = NATIVE_PX_M / M_PER_DEG_LAT
    dlon = NATIVE_PX_M / (M_PER_DEG_LON_EQ * np.cos(np.radians(lat0)))
    for rot in [0.0, -12.0, 15.0, 168.0]:   # ~ascending, descending-like headings
        th = np.radians(rot)
        # pixel (x right, y down) -> (east, north) meters, rotated clockwise by rot:
        #   east  =  cos(th)*x - sin(th)*y
        #   north = -sin(th)*x - cos(th)*y
        # (a vector at image bearing b then lands at geographic bearing b+rot)
        a_e, b_e = np.cos(th) * NATIVE_PX_M, -np.sin(th) * NATIVE_PX_M
        d_n, e_n = -np.sin(th) * NATIVE_PX_M, -np.cos(th) * NATIVE_PX_M
        T = Affine(a_e / NATIVE_PX_M * dlon, b_e / NATIVE_PX_M * dlon, lon0,
                   d_n / NATIVE_PX_M * dlat, e_n / NATIVE_PX_M * dlat, lat0)
        for img_brg in [0.0, 70.0, 225.0]:
            r = extract_dominant_peak(make_synthetic_stripe(256, 256, 40, img_brg), 1.0)
            ga, gb = geographic_bearing(T, 1000, 1000, r["freq_y"], r["freq_x"])
            expected = (img_brg + rot) % 360.0
            ok = min(angular_diff(ga, expected), angular_diff(gb, expected)) < 10
            ok_all &= ok
            print(f"  {'PASS' if ok else 'FAIL'}  image rot {rot:+6.1f}deg, image bearing "
                  f"{img_brg:5.1f} -> geo {ga:5.1f}/{gb:5.1f} (expected {expected:5.1f})")

    print("\n[3] Realistic packets: rank-ordered solitons inside a cluttered crop")
    print("    (red-noise background, range ramp, dark slick, speckle -- the setting")
    print("     where v1 locked onto crop-scale trends on real data)")
    rng = np.random.default_rng(11)
    n = n_rel = rel_ok = stuck = 0
    for sp_m in [600, 900, 1200, 1500, 2000]:
        for brg in np.linspace(0, 330, 12):
            for (h, w) in [(700, 500), (560, 480), (900, 400)]:
                img = make_synthetic_packet(h, w, sp_m / PIXEL_SCALE_M, brg, rng)
                r = extract_dominant_peak(img, PIXEL_SCALE_M)
                n += 1
                stuck += r["wavelength_m"] > 0.3 * min(h, w) * PIXEL_SCALE_M
                if r["spectral_peak_snr"] >= SNR_RELIABLE and r["periods_in_crop"] >= PERIODS_RELIABLE:
                    n_rel += 1
                    d = abs(r["image_bearing_deg"] - brg) % 180
                    rel_ok += (min(d, 180 - d) < 15
                               and abs(r["wavelength_m"] - sp_m) / sp_m < 0.25)
    acc = rel_ok / max(n_rel, 1)
    ok3 = acc >= 0.8 and stuck / n < 0.1 and n_rel >= 0.25 * n
    ok_all &= ok3
    print(f"  {'PASS' if ok3 else 'FAIL'}  peaks stuck at crop scale: {stuck}/{n} "
          f"| flagged reliable: {n_rel}/{n} | reliable & correct: {rel_ok}/{n_rel} ({acc:.0%})")

    print("\n[4] Packet + strong monsoon swell (300 m, crop-filling) in the same crop")
    n = n_rel = rel_ok = swell_seen = 0
    for sp_m in [800, 1200, 1500]:
        for brg in np.linspace(0, 330, 12):
            img = make_synthetic_packet(700, 500, sp_m / PIXEL_SCALE_M, brg, rng,
                                        swell=(0.5, 300 / PIXEL_SCALE_M, brg + 40))
            r = extract_dominant_peak(img, PIXEL_SCALE_M)
            n += 1
            swell_seen += r["swell_snr"] >= SNR_RELIABLE
            if r["spectral_peak_snr"] >= SNR_RELIABLE and r["periods_in_crop"] >= PERIODS_RELIABLE:
                n_rel += 1
                d = abs(r["image_bearing_deg"] - brg) % 180
                rel_ok += (min(d, 180 - d) < 15
                           and abs(r["wavelength_m"] - sp_m) / sp_m < 0.25)
    acc = rel_ok / max(n_rel, 1)
    ok4 = acc >= 0.8 and swell_seen >= 0.8 * n and n_rel >= 0.25 * n
    ok_all &= ok4
    print(f"  {'PASS' if ok4 else 'FAIL'}  swell detected in diagnostic: {swell_seen}/{n} "
          f"| flagged reliable: {n_rel}/{n} | reliable & packet correct: {rel_ok}/{n_rel} ({acc:.0%})")

    print("\n[5] Red-heavy spectra (stronger, steeper background) -- harmonic check must")
    print("    not drag estimates to the long-wavelength limit (the v2 real-data failure)")
    n = harm = at_lim = n_rel = rel_ok = 0
    for sp_m in [800, 1200, 1500, 2000]:
        for brg in np.linspace(0, 330, 12):
            h, w = 700, 500
            img = make_synthetic_packet(h, w, sp_m / PIXEL_SCALE_M, brg, rng,
                                        red_amp=1.2, red_beta=2.6)
            r = extract_dominant_peak(img, PIXEL_SCALE_M)
            n += 1
            harm += r["harmonic_n"] > 1
            at_lim += r["wavelength_m"] > 0.9 * min(h, w) / MIN_PERIODS_IN_CROP * PIXEL_SCALE_M
            if r["spectral_peak_snr"] >= SNR_RELIABLE and r["periods_in_crop"] >= PERIODS_RELIABLE:
                n_rel += 1
                d = abs(r["image_bearing_deg"] - brg) % 180
                rel_ok += (min(d, 180 - d) < 15
                           and abs(r["wavelength_m"] - sp_m) / sp_m < 0.25)
    acc = rel_ok / max(n_rel, 1)
    ok5 = harm <= 0.15 * n and at_lim <= 0.15 * n and acc >= 0.8
    ok_all &= ok5
    print(f"  {'PASS' if ok5 else 'FAIL'}  harmonic jumps: {harm}/{n} | at long limit: {at_lim}/{n} "
          f"| reliable & correct: {rel_ok}/{n_rel} ({acc:.0%})")

    print("\n" + "=" * 70)
    print("ALL SELF-TESTS PASSED." if ok_all else "SELF-TEST FAILURES -- do not run on real data.")
    print("=" * 70)
    return ok_all


# ----------------------------------------------------------------------
# Real run
# ----------------------------------------------------------------------
def run():
    det_path, tiles_dir = Path(DETECTIONS_PARQUET), Path(TILES_DIR)
    for p, what in [(det_path, "DETECTIONS_PARQUET"), (tiles_dir, "TILES_DIR"),
                    (Path(SAFE_ROOT), "SAFE_ROOT")]:
        if not p.exists():
            print(f"ERROR: {what} not found: {p}")
            return

    det = pd.read_parquet(det_path)
    print(f"Loaded {len(det)} detections. Columns: {list(det.columns)}")
    print(f"Tile scale: {PIXEL_SCALE_M:.2f} m/px "
          f"({TILE_SIZE_KM} km tiles resampled to {OUTPUT_PIXELS}px)")

    # Land mask: older parquets were written before land-masking existed
    # (the original 798-detection run had ~450 detections on land). Always
    # re-apply it here so Tier 1 never measures rivers/deltas.
    if not Path(LAND_MASK_PATH).exists():
        print(f"ERROR: LAND_MASK_PATH not found: {LAND_MASK_PATH}")
        return
    is_on_land = load_land_checker(LAND_MASK_PATH)
    on_land = det.apply(lambda r: is_on_land(float(r["real_lat"]), float(r["real_lon"])), axis=1)
    print(f"Land mask: dropping {int(on_land.sum())} of {len(det)} detections on land")
    det = det[~on_land].reset_index(drop=True)

    # Tile filename: present in newer parquets, rebuilt from scene_id + tile_idx otherwise
    if "tile_filename" not in det.columns:
        lookup = build_tile_lookup(tiles_dir)
        print(f"No 'tile_filename' column -- matched via scene_id + tile_idx "
              f"against {len(lookup)} tiles in TILES_DIR")
        det["tile_filename"] = [lookup.get((s, int(t))) for s, t in
                                zip(det["scene_id"], det["tile_idx"])]

    tile_cache, transform_cache = {}, {}
    results = []
    n_missing_tile = n_bad_crop = n_no_transform = n_low = 0

    for i, (_, row) in enumerate(det.iterrows()):
        fname = row["tile_filename"]
        if fname is None:
            n_missing_tile += 1
            continue
        if fname not in tile_cache:
            tp = tiles_dir / fname
            tile_cache[fname] = np.array(Image.open(tp).convert("L")) if tp.exists() else None
        tile = tile_cache[fname]
        if tile is None:
            n_missing_tile += 1
            continue

        x1, y1, x2, y2 = get_box_xyxy(row)
        crop = crop_box(tile, x1, y1, x2, y2, CROP_PAD_PX)
        peak = extract_dominant_peak(crop, PIXEL_SCALE_M)
        if peak is None:
            n_bad_crop += 1
            continue

        scene_id = row["scene_id"]
        if scene_id not in transform_cache:
            try:
                transform_cache[scene_id] = load_scene_transform(scene_id)
            except Exception as e:
                print(f"  WARNING: no GCP transform for {scene_id}: {e}")
                transform_cache[scene_id] = None
        T = transform_cache[scene_id]

        if T is not None:
            geo_a, geo_b = geographic_bearing(
                T, row["scene_pixel_x"], row["scene_pixel_y"],
                peak["freq_y"], peak["freq_x"],
            )
        else:
            geo_a = geo_b = np.nan
            n_no_transform += 1

        low = (not np.isfinite(peak["spectral_peak_snr"])
               or peak["spectral_peak_snr"] < SNR_RELIABLE
               or peak["periods_in_crop"] < PERIODS_RELIABLE
               or T is None)
        n_low += int(low)

        results.append({
            "scene_id": scene_id,
            "tile_filename": fname,
            "real_lat": row.get("real_lat"),
            "real_lon": row.get("real_lon"),
            "confidence": row.get("confidence"),
            "crop_h": crop.shape[0], "crop_w": crop.shape[1],
            "wavelength_m": peak["wavelength_m"],
            "image_bearing_deg": peak["image_bearing_deg"],
            "bearing_deg_a": geo_a,
            "bearing_deg_b": geo_b,
            "spectral_peak_snr": peak["spectral_peak_snr"],
            "periods_in_crop": peak["periods_in_crop"],
            "harmonic_n": peak["harmonic_n"],
            "swell_snr": peak["swell_snr"],
            "swell_wavelength_m": peak["swell_wavelength_m"],
            "low_confidence": low,
        })
        if (i + 1) % 100 == 0:
            print(f"  processed {i + 1}/{len(det)}")

    out = pd.DataFrame(results)
    out.to_parquet(OUTPUT_PARQUET, index=False)

    print(f"\n{'=' * 60}\nTIER 1 SUMMARY\n{'=' * 60}")
    print(f"Detections in:              {len(det)}")
    print(f"Processed:                  {len(out)}")
    print(f"Tile file not found:        {n_missing_tile}")
    print(f"Crop too small/degenerate:  {n_bad_crop}")
    print(f"No GCP transform (bearing NaN): {n_no_transform}")
    print(f"Low-confidence flagged:     {n_low} ({100 * n_low / max(1, len(out)):.1f}%)"
          f"  [SNR < {SNR_RELIABLE} or < {PERIODS_RELIABLE} periods in crop]")
    if len(out):
        rel = out[~out["low_confidence"]]
        print(f"Reliable (go to Tier 2):    {len(rel)}")
        print("\nWavelength (m), reliable only:")
        print(rel["wavelength_m"].describe().round(1).to_string())
        print("\nWavelength (m), all:")
        print(out["wavelength_m"].describe().round(1).to_string())
        print("\nWhitened spectral SNR (all):")
        print(out["spectral_peak_snr"].describe().round(1).to_string())
        print("\nPeriods in crop (all):")
        print(out["periods_in_crop"].describe().round(1).to_string())
        print(f"\nHarmonic correction applied: {(out['harmonic_n'] > 1).sum()}")
        lim = out[["crop_h", "crop_w"]].min(axis=1) / MIN_PERIODS_IN_CROP * PIXEL_SCALE_M
        print(f"At long-wavelength search limit (>90%): {(out['wavelength_m'] > 0.9 * lim).sum()}")
        print(f"At 500 m search floor (<550 m):         {(out['wavelength_m'] < 550).sum()}")
        print(f"Swell present (100-500 m, SNR >= {SNR_RELIABLE:.0f}): "
              f"{(out['swell_snr'] >= SNR_RELIABLE).sum()}  "
              f"(diagnostic only -- not used for Tier 2)")
    print(f"\nSaved to {OUTPUT_PARQUET}")


def inspect(n_show=12, out_png="tier1_inspect.png"):
    """Visual check: saves a grid of crops (the reliable detections with the
    highest SNR) with the estimated crest-normal axis drawn and the
    wavelength/SNR in the title. Run after 'run'."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    res = pd.read_parquet(OUTPUT_PARQUET)
    det = pd.read_parquet(DETECTIONS_PARQUET)
    if "tile_filename" not in det.columns:
        lookup = build_tile_lookup(TILES_DIR)
        det["tile_filename"] = [lookup.get((s, int(t))) for s, t in
                                zip(det["scene_id"], det["tile_idx"])]
    det = det.drop_duplicates(subset=["tile_filename", "real_lat", "real_lon"])
    merged = res.merge(det[["tile_filename", "real_lat", "real_lon", "box_resized"]],
                       on=["tile_filename", "real_lat", "real_lon"], how="left")
    pick = (merged[~merged["low_confidence"]]
            .sort_values("spectral_peak_snr", ascending=False).head(n_show))
    if pick.empty:
        print("No reliable detections to show.")
        return

    cols = 4
    rows = int(np.ceil(len(pick) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    for ax, (_, r) in zip(np.atleast_1d(axes).ravel(), pick.iterrows()):
        tile = np.array(Image.open(Path(TILES_DIR) / r["tile_filename"]).convert("L"))
        x1, y1, x2, y2 = np.asarray(r["box_resized"], dtype=float)
        crop = crop_box(tile, x1, y1, x2, y2, CROP_PAD_PX)
        ax.imshow(crop, cmap="gray")
        h, w = crop.shape
        b = np.radians(r["image_bearing_deg"])
        L = 0.4 * min(h, w)
        ax.plot([w / 2 - L * np.sin(b), w / 2 + L * np.sin(b)],
                [h / 2 + L * np.cos(b), h / 2 - L * np.cos(b)], "r-", lw=2)
        ax.set_title(f"{r['wavelength_m']:.0f} m | SNR {r['spectral_peak_snr']:.0f} | "
                     f"{r['periods_in_crop']:.0f} per.\n{r['real_lat']:.2f}N {r['real_lon']:.2f}E",
                     fontsize=8)
        ax.axis("off")
    for ax in np.atleast_1d(axes).ravel()[len(pick):]:
        ax.axis("off")
    fig.suptitle("Tier 1: red line = estimated propagation axis\n"
                 "(should cross the crests at right angles)", fontsize=10)
    fig.tight_layout()
    fig.savefig(out_png, dpi=110)
    print(f"Saved {out_png} ({len(pick)} crops)")


def main():
    # In a notebook, sys.argv holds Jupyter's own kernel args -- ignore them
    # and use MODE. From a terminal, a 'selftest'/'run' argument wins.
    in_notebook = "ipykernel" in sys.modules
    mode = MODE
    if not in_notebook and len(sys.argv) == 2 and sys.argv[1] in ("selftest", "run", "inspect"):
        mode = sys.argv[1]
    print(f"Mode: {mode}")
    if mode == "selftest":
        selftest()
    elif mode == "run":
        run()
    elif mode == "inspect":
        inspect()
    else:
        print(f"Unknown MODE {mode!r}; use 'selftest', 'run' or 'inspect'.")


if __name__ == "__main__":
    main()

Mode: inspect
Saved tier1_inspect.png (12 crops)


In [10]:
""" V4
OceanXRay — Tier 1: FFT-based wavelength & propagation-direction extraction
=============================================================================
For each land-filtered internal-wave detection in
real_internal_wave_detections.parquet (written by infer_iw_on_real_scenes.py):

  1. Load the source SAR tile the detection came from.
  2. Crop a padded window around the detection box (`box_resized`,
     [x1, y1, x2, y2] in the 1024x1024 tile's pixel space).
  3. Remove the large-scale brightness plane, apply a 2D Hann window.
  4. 2D FFT, divide by a fitted red-noise background, and take the strongest
     directional peak among wavelengths 250 m .. crop/3 (with a harmonic
     check). v1 skipped the background step and on real data latched onto
     crop-scale brightness trends instead of soliton spacing -- see
     extract_dominant_peak.
     Detections are flagged low_confidence unless SNR >= 20 (above the
     99th percentile of wave-free crops) and >= 5 periods fit in the crop.
  5. Convert the peak to:
       - wavelength in meters (tile pixel scale ~24.4 m/px)
       - propagation direction, first in IMAGE space, then converted to a
         real GEOGRAPHIC bearing (0=N, 90=E) using the scene's own GCP
         transform. Sentinel-1 GRD images are in radar (slant/ground-range)
         geometry, NOT north-up -- an image-space "up" is the satellite's
         flight direction, which is rotated ~10-15 deg off north and
         flipped between ascending and descending passes. Using image-space
         bearings directly would aim every Tier 2 ray in the wrong direction.
       - Both 180-deg-apart candidates are kept (bearing_deg_a, bearing_deg_b):
         a real-valued FFT is Hermitian-symmetric, so it physically cannot
         tell "this way" from "opposite way". Tier 2 resolves it.

HOW TO RUN
----------
Kaggle/Colab notebook cell (paste or %run): set MODE below to "selftest"
first, run it, then MODE = "run", then MODE = "review" (saves numbered
picture sheets + review_labels.csv to fill in by eye), then MODE = "score".

Terminal:
    python tier1_fft_backtrack.py selftest
    python tier1_fft_backtrack.py run

Output: tier1_fft_results.parquet
"""

import sys
import glob
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

# ======================= EDIT THESE =======================
MODE = "score"   # "selftest" -> "run" -> "review" -> (fill CSV) -> "score"

DETECTIONS_PARQUET = "/kaggle/input/datasets/mayankgupta005/internal-wave-detection-metadata/real_internal_wave_detections.parquet"
TILES_DIR = "/kaggle/input/datasets/mayankgupta005/sar-image-tiles-june-2021/sar_images_for_iw_detection"          # same folder as IMAGE_DIR in infer_iw_on_real_scenes.py
SAFE_ROOT = "/kaggle/input/datasets/andrejkarpathy/sentinel-1-june-2021-raw-safe"       # same folder as SAFE_ROOT in infer_iw_on_real_scenes.py
OUTPUT_PARQUET = "tier1_fft_results.parquet"
LAND_MASK_PATH = "/kaggle/input/datasets/mayankgupta005/internal-wave-detection-metadata/land_ocean_mask.npz"  # same file infer_iw_on_real_scenes.py uses

# These two are confirmed from convert_scenes_to_images.py
# (TILE_SIZE_KM = 25, OUTPUT_PIXELS = 1024) -- no need to change them
# unless you changed those settings there.
TILE_SIZE_KM = 25
OUTPUT_PIXELS = 1024
# ==========================================================

NATIVE_PX_M = 10.0                                    # S1 IW GRD pixel spacing
NATIVE_TILE_PX = int(TILE_SIZE_KM * 1000 / NATIVE_PX_M)  # 2500
TILE_TO_SCENE = NATIVE_TILE_PX / OUTPUT_PIXELS        # 1 tile px = 2.44 scene px
PIXEL_SCALE_M = TILE_SIZE_KM * 1000 / OUTPUT_PIXELS   # ~24.4 m per tile px

CROP_PAD_PX = 40
MIN_CROP_SIDE_PX = 48
COAST_BUFFER_KM = 1.0   # drop detections whose box, grown by this much, touches land

# v2 spectral search settings (see extract_dominant_peak docstring)
# Search floor 500 m: ocean swell (8-17 s periods -> ~100-450 m wavelength,
# strongest during the June SW monsoon) is the main periodic confounder in
# SAR. The v2 run on real data put 49/105 'reliable' results at 250-500 m
# with a median ~34 periods per crop -- a crop-filling texture, i.e. swell,
# not a soliton packet (IW Atlas, Andaman Sea: soliton surface width
# 600-1000 m). Swell energy is still measured and reported separately.
LAMBDA_MIN_PX = 500.0 / PIXEL_SCALE_M
SWELL_BAND_M = (100.0, 500.0)
MIN_PERIODS_IN_CROP = 3.0               # search only wavelengths fitting >=3 times in the crop
HARMONIC_FRAC = 0.35                    # sub-harmonic must hold >=35% of peak's whitened power

# Reliability flags. SNR = peak power / fitted red-noise background.
# SNR_RELIABLE = 20 is the 99th percentile of SNR on wave-free crops (pure
# red noise + range ramp + speckle: median 6.8, 99th pct 20.0, max 22.1).
# On synthetic rank-ordered packets in heavy clutter, crops passing BOTH
# flags had direction and wavelength right 159/159; the rest ~59% / ~51%.
SNR_RELIABLE = 20.0
PERIODS_RELIABLE = 5.0

M_PER_DEG_LAT = 110_540.0
M_PER_DEG_LON_EQ = 111_320.0


# ----------------------------------------------------------------------
# Core FFT routine
# ----------------------------------------------------------------------
def image_bearing_from_freq(fy, fx):
    """Bearing in IMAGE space (0 = image-up, 90 = image-right)."""
    return np.degrees(np.arctan2(fx, -fy)) % 360.0


def extract_dominant_peak(crop, pixel_scale_m):
    """
    v2 spectral peak finder. Returns dict (incl. raw freq_y/freq_x) or None.

    v1 (plain FFT argmax) failed on real data: SAR spectra are 'red' (power
    rises steeply toward low frequency), so the argmax always landed on the
    first bin outside the DC exclusion -- 232/346 real detections peaked at
    exactly 2.2 bins, giving 'wavelengths' of ~half the crop size. v2 fixes:
      1. Remove a best-fit plane (incidence-angle ramp / large brightness trend).
      2. Whiten: divide each bin by the mean power of its frequency annulus,
         so the peak is a DIRECTIONAL excess over the background at that
         scale, not just 'the lowest frequency available'.
      3. Search only wavelengths in [250 m, crop/3] -- longer can't be
         measured from a crop holding <3 periods.
      4. Harmonic check: sharp sech^2 solitons put strong energy at 2x-5x the
         fundamental; if a sub-harmonic (f/n) in the same direction holds
         >= HARMONIC_FRAC of the peak power, take it as the fundamental.
    spectral_peak_snr is the whitened peak power (~1 = background level).
    """
    h, w = crop.shape
    if h < MIN_CROP_SIDE_PX or w < MIN_CROP_SIDE_PX:
        return None

    c = crop.astype(np.float64)
    yy, xx = np.mgrid[0:h, 0:w]
    A = np.c_[np.ones(h * w), xx.ravel(), yy.ravel()]
    coef, *_ = np.linalg.lstsq(A, c.ravel(), rcond=None)
    c = (c - (A @ coef).reshape(h, w)) * np.outer(np.hanning(h), np.hanning(w))

    power = np.abs(np.fft.fftshift(np.fft.fft2(c))) ** 2
    cy, cx = h // 2, w // 2
    fy = (yy - cy) / h
    fx = (xx - cx) / w
    fr = np.hypot(fy, fx)

    with np.errstate(divide="ignore"):
        lam_px = np.where(fr > 0, 1.0 / fr, np.inf)
    band = (lam_px >= LAMBDA_MIN_PX) & (lam_px <= min(h, w) / MIN_PERIODS_IN_CROP)
    if not band.any():
        return None

    # Background spectrum: median power per frequency annulus (a few strong
    # signal bins can't move a median), then a smooth power-law fit across
    # radius, so the background is the red-noise trend, not the signal itself.
    rbin = np.round(fr * min(h, w)).astype(int)
    radii = np.unique(rbin[band])
    medians = np.array([np.median(power[(rbin == r) & band]) for r in radii])
    good = (radii > 0) & (medians > 0)
    if good.sum() >= 3:
        slope, icpt = np.polyfit(np.log(radii[good]), np.log(medians[good]), 1)
        background = np.exp(icpt + slope * np.log(np.maximum(rbin, 1)))
    else:
        background = np.full_like(power, np.median(power[band]))
    whitened = power / np.maximum(background, 1e-30)
    searched = np.where(band, whitened, -np.inf)

    py, px = np.unravel_index(np.argmax(searched), searched.shape)
    peak_snr = whitened[py, px]
    best_y, best_x, best_snr, harmonic_n = py, px, peak_snr, 1
    # Harmonic check, n = 2, 3 only. Accept a sub-harmonic only if it is a
    # genuine spectral peak on its own: a strict local maximum of the
    # whitened spectrum AND above the reliability SNR. (An earlier
    # 'raw power >= 50%' rule fired on 61% of real crops and dragged
    # estimates to the long-wavelength limit, because real SAR spectra
    # always carry more raw power at lower frequencies.)
    from scipy.ndimage import maximum_filter
    local_max = whitened >= maximum_filter(whitened, size=5)
    for n in (2, 3):
        y0 = int(round(cy + (py - cy) / n))
        x0 = int(round(cx + (px - cx) / n))
        ys = slice(max(0, y0 - 1), y0 + 2)
        xs = slice(max(0, x0 - 1), x0 + 2)
        sub = np.where(local_max[ys, xs], searched[ys, xs], -np.inf)
        if not (sub.size and np.isfinite(sub).any()):
            continue
        if sub.max() >= max(HARMONIC_FRAC * peak_snr, SNR_RELIABLE):
            iy, ix = np.unravel_index(np.argmax(sub), sub.shape)
            best_y, best_x = ys.start + iy, xs.start + ix
            best_snr, harmonic_n = whitened[best_y, best_x], n

    # Swell diagnostic: strongest peak in the 100-500 m band, measured against
    # that band's OWN background fit (speckle flattens the spectrum at these
    # scales, so extrapolating the main-band fit would inflate every bin).
    swell_band = ((lam_px >= SWELL_BAND_M[0] / pixel_scale_m)
                  & (lam_px < SWELL_BAND_M[1] / pixel_scale_m))
    swell_snr, swell_wavelength_m = np.nan, np.nan
    s_radii = np.unique(rbin[swell_band])
    if s_radii.size >= 3:
        s_med = np.array([np.median(power[(rbin == r) & swell_band]) for r in s_radii])
        ok = (s_radii > 0) & (s_med > 0)
        if ok.sum() >= 3:
            s_slope, s_icpt = np.polyfit(np.log(s_radii[ok]), np.log(s_med[ok]), 1)
            s_bg = np.exp(s_icpt + s_slope * np.log(np.maximum(rbin, 1)))
            s_white = np.where(swell_band, power / np.maximum(s_bg, 1e-30), -np.inf)
            sy, sx = np.unravel_index(np.argmax(s_white), s_white.shape)
            swell_snr = float(s_white[sy, sx])
            swell_wavelength_m = float(pixel_scale_m / max(fr[sy, sx], 1e-12))

    freq_y, freq_x = fy[best_y, best_x], fx[best_y, best_x]
    freq_mag = np.hypot(freq_y, freq_x)
    if freq_mag <= 0:
        return None
    wavelength_px = 1.0 / freq_mag

    return {
        "freq_y": float(freq_y),
        "freq_x": float(freq_x),
        "wavelength_m": float(wavelength_px * pixel_scale_m),
        "image_bearing_deg": float(image_bearing_from_freq(freq_y, freq_x)),
        "spectral_peak_snr": float(best_snr),
        "periods_in_crop": float(min(h, w) / wavelength_px),
        "harmonic_n": int(harmonic_n),
        "swell_snr": swell_snr,
        "swell_wavelength_m": swell_wavelength_m,
    }


# ----------------------------------------------------------------------
# Image-space -> geographic bearing via the scene's GCP transform
# ----------------------------------------------------------------------
def geographic_bearing(transform, cx_scene, cy_scene, freq_y, freq_x, half_len_px=100.0):
    """
    Convert an image-space wavevector to a geographic propagation bearing.

    We transform the CREST direction (perpendicular to the wavevector),
    not the wavevector itself: a crest line is a physical displacement
    and maps correctly through the pixel->lon/lat transform, whereas a
    normal vector does not under a non-conformal mapping (lon/lat degrees
    are not equal-length). The propagation bearing is then the crest
    tangent rotated 90 deg, measured in local east/north METERS.
    Returns (bearing_a, bearing_b) -- 180 deg apart.
    """
    k = np.array([freq_x, freq_y])          # (x=col, y=row)
    k = k / np.linalg.norm(k)
    t = np.array([-k[1], k[0]])             # crest tangent in pixel space

    p1 = (cx_scene - half_len_px * t[0], cy_scene - half_len_px * t[1])
    p2 = (cx_scene + half_len_px * t[0], cy_scene + half_len_px * t[1])
    lon1, lat1 = transform * p1
    lon2, lat2 = transform * p2

    mean_lat = np.radians((lat1 + lat2) / 2)
    east = (lon2 - lon1) * M_PER_DEG_LON_EQ * np.cos(mean_lat)
    north = (lat2 - lat1) * M_PER_DEG_LAT

    crest_bearing = np.degrees(np.arctan2(east, north)) % 360.0
    bearing_a = (crest_bearing + 90.0) % 360.0
    bearing_b = (crest_bearing + 270.0) % 360.0
    return float(bearing_a), float(bearing_b)


# ----------------------------------------------------------------------
# Scene transform loading (same method as infer_iw_on_real_scenes.py)
# ----------------------------------------------------------------------
def find_measurement_tiff(safe_dir, polarization="vv"):
    matches = glob.glob(os.path.join(safe_dir, "measurement", f"*{polarization}*.tiff"))
    if not matches:
        matches = glob.glob(os.path.join(safe_dir, "**", "measurement",
                                         f"*{polarization}*.tiff"), recursive=True)
    if not matches:
        raise FileNotFoundError(f"No {polarization} measurement GeoTIFF under {safe_dir}")
    return matches[0]


def load_scene_transform(scene_id):
    import rasterio
    from rasterio.transform import from_gcps
    tiff = find_measurement_tiff(os.path.join(SAFE_ROOT, f"{scene_id}.SAFE"))
    with rasterio.open(tiff) as src:
        gcps, _ = src.get_gcps()
        if not gcps:
            raise ValueError(f"No GCPs in {tiff}")
        return from_gcps(gcps)


# ----------------------------------------------------------------------
# Tile-filename lookup + land mask (for parquets from older inference runs)
# ----------------------------------------------------------------------
TILE_FILENAME_RE = re.compile(r"^(.+)_tile(\d+)_row(\d+)_col(\d+)\.png$")


def build_tile_lookup(tiles_dir):
    """Map (scene_id, tile_idx) -> tile filename by scanning TILES_DIR.
    Older parquets only store scene_id + tile_idx, not the filename."""
    lookup = {}
    for p in Path(tiles_dir).glob("*.png"):
        m = TILE_FILENAME_RE.match(p.name)
        if m:
            lookup[(m.group(1), int(m.group(2)))] = p.name
    return lookup


def load_land_checker(path):
    """Same lookup as infer_iw_on_real_scenes.py's is_on_land."""
    d = np.load(path)
    mask, mlat, mlon = d["mask"], d["lat"], d["lon"]

    def is_on_land(lat, lon):
        lat = max(mlat.min(), min(mlat.max(), lat))
        lon = max(mlon.min(), min(mlon.max(), lon))
        i = int((lat - mlat[0]) / (mlat[1] - mlat[0]))
        j = int((lon - mlon[0]) / (mlon[1] - mlon[0]))
        return not bool(mask[i, j])   # mask True = ocean
    return is_on_land


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------
def get_box_xyxy(row):
    """Read the detection box as [x1, y1, x2, y2] in 1024-tile pixels."""
    if "box_resized" in row.index and row["box_resized"] is not None:
        b = np.asarray(row["box_resized"], dtype=float)
        return b[0], b[1], b[2], b[3]
    if {"bbox_x", "bbox_y", "bbox_w", "bbox_h"}.issubset(row.index):
        return (row["bbox_x"], row["bbox_y"],
                row["bbox_x"] + row["bbox_w"], row["bbox_y"] + row["bbox_h"])
    raise KeyError("No box column found (expected 'box_resized')")


def crop_box(tile, x1, y1, x2, y2, pad):
    h, w = tile.shape
    return tile[max(0, int(y1) - pad):min(h, int(y2) + pad),
                max(0, int(x1) - pad):min(w, int(x2) + pad)]


def angular_diff(a, b):
    d = abs(a - b) % 360.0
    return min(d, 360.0 - d)


def make_synthetic_stripe(h, w, wavelength_px, image_bearing_deg, noise_std=0.05, seed=0):
    rng = np.random.default_rng(seed)
    fx = np.sin(np.radians(image_bearing_deg))
    fy = -np.cos(np.radians(image_bearing_deg))
    yy, xx = np.mgrid[0:h, 0:w]
    img = np.sin(2 * np.pi * (fx * xx + fy * yy) / wavelength_px)
    return img + rng.normal(0, noise_std, size=img.shape)


def _red_noise(h, w, rng, beta=2.0):
    fy = np.fft.fftfreq(h)[:, None]
    fx = np.fft.fftfreq(w)[None, :]
    f = np.hypot(fy, fx)
    f[0, 0] = 1
    spec = (rng.normal(size=(h, w)) + 1j * rng.normal(size=(h, w))) / f ** (beta / 2)
    n = np.real(np.fft.ifft2(spec))
    return (n - n.mean()) / n.std()


def make_synthetic_packet(h, w, spacing_px, image_bearing_deg, rng,
                          red_amp=0.75, red_beta=2.0, swell=None):
    """Rank-ordered sech^2 soliton packet occupying part of the crop, on a
    red-noise background with a range ramp, a dark slick and 4-look speckle.
    swell=(amplitude, wavelength_px, image_bearing_deg) adds a crop-filling
    swell field."""
    yy, xx = np.mgrid[0:h, 0:w]
    ex, ny = np.sin(np.radians(image_bearing_deg)), np.cos(np.radians(image_bearing_deg))
    s = ex * (xx - w / 2) - ny * (yy - h / 2)          # along propagation
    t = ny * (xx - w / 2) + ex * (yy - h / 2)          # along crest
    L = 0.8 * min(h, w)
    sig = np.zeros((h, w))
    for i in range(max(3, int(L / spacing_px))):
        sig += (1.0 - 0.12 * i) / np.cosh((s - (L / 2 - i * spacing_px)) / (spacing_px / 5)) ** 2
    envelope = np.exp(-(t / (0.45 * max(h, w))) ** 4) * (np.abs(s) < L / 2 + spacing_px)
    slick = -0.8 * np.exp(-(((xx - 0.3 * w) / (0.35 * w)) ** 2 + ((yy - 0.7 * h) / (0.3 * h)) ** 2))
    img = (1.0 + 0.7 * sig * envelope + red_amp * _red_noise(h, w, rng, red_beta)
           + 1.2 * xx / w + slick)
    if swell is not None:
        s_amp, s_wl, s_brg = swell
        s_e, s_n = np.sin(np.radians(s_brg)), np.cos(np.radians(s_brg))
        img += s_amp * np.sin(2 * np.pi * (s_e * xx - s_n * yy) / s_wl)
    return np.clip(img, 0.05, None) * rng.gamma(4, 0.25, size=(h, w))


# ----------------------------------------------------------------------
# Self-test
# ----------------------------------------------------------------------
def selftest():
    from affine import Affine
    print("=" * 70)
    print("TIER 1 SELF-TEST")
    print("=" * 70)
    ok_all = True

    print("\n[1] Wavelength + image-space bearing on synthetic stripes")
    # (all >= the 500 m / ~20.5 px search floor)
    for wl, brg in [(30, 0), (30, 90), (45, 135), (60, 200), (25, 315)]:
        r = extract_dominant_peak(make_synthetic_stripe(256, 256, wl, brg), 1.0)
        b_a = r["image_bearing_deg"]
        ok = (abs(r["wavelength_m"] - wl) < 3.5
              and min(angular_diff(b_a, brg), angular_diff(b_a + 180, brg)) < 10)
        ok_all &= ok
        print(f"  {'PASS' if ok else 'FAIL'}  true {wl}px @ {brg}deg -> "
              f"{r['wavelength_m']:.1f}px @ {b_a:.1f}/{(b_a+180)%360:.1f}deg")

    print("\n[2] Image -> geographic bearing through a rotated GCP-style transform")
    # Build pixel->lon/lat transforms like a real S1 scene: 10 m pixels,
    # image rotated by `rot` deg relative to north, at lat ~12N.
    lat0, lon0 = 12.0, 90.0
    dlat = NATIVE_PX_M / M_PER_DEG_LAT
    dlon = NATIVE_PX_M / (M_PER_DEG_LON_EQ * np.cos(np.radians(lat0)))
    for rot in [0.0, -12.0, 15.0, 168.0]:   # ~ascending, descending-like headings
        th = np.radians(rot)
        # pixel (x right, y down) -> (east, north) meters, rotated clockwise by rot:
        #   east  =  cos(th)*x - sin(th)*y
        #   north = -sin(th)*x - cos(th)*y
        # (a vector at image bearing b then lands at geographic bearing b+rot)
        a_e, b_e = np.cos(th) * NATIVE_PX_M, -np.sin(th) * NATIVE_PX_M
        d_n, e_n = -np.sin(th) * NATIVE_PX_M, -np.cos(th) * NATIVE_PX_M
        T = Affine(a_e / NATIVE_PX_M * dlon, b_e / NATIVE_PX_M * dlon, lon0,
                   d_n / NATIVE_PX_M * dlat, e_n / NATIVE_PX_M * dlat, lat0)
        for img_brg in [0.0, 70.0, 225.0]:
            r = extract_dominant_peak(make_synthetic_stripe(256, 256, 40, img_brg), 1.0)
            ga, gb = geographic_bearing(T, 1000, 1000, r["freq_y"], r["freq_x"])
            expected = (img_brg + rot) % 360.0
            ok = min(angular_diff(ga, expected), angular_diff(gb, expected)) < 10
            ok_all &= ok
            print(f"  {'PASS' if ok else 'FAIL'}  image rot {rot:+6.1f}deg, image bearing "
                  f"{img_brg:5.1f} -> geo {ga:5.1f}/{gb:5.1f} (expected {expected:5.1f})")

    print("\n[3] Realistic packets: rank-ordered solitons inside a cluttered crop")
    print("    (red-noise background, range ramp, dark slick, speckle -- the setting")
    print("     where v1 locked onto crop-scale trends on real data)")
    rng = np.random.default_rng(11)
    n = n_rel = rel_ok = stuck = 0
    for sp_m in [600, 900, 1200, 1500, 2000]:
        for brg in np.linspace(0, 330, 12):
            for (h, w) in [(700, 500), (560, 480), (900, 400)]:
                img = make_synthetic_packet(h, w, sp_m / PIXEL_SCALE_M, brg, rng)
                r = extract_dominant_peak(img, PIXEL_SCALE_M)
                n += 1
                stuck += r["wavelength_m"] > 0.3 * min(h, w) * PIXEL_SCALE_M
                if r["spectral_peak_snr"] >= SNR_RELIABLE and r["periods_in_crop"] >= PERIODS_RELIABLE:
                    n_rel += 1
                    d = abs(r["image_bearing_deg"] - brg) % 180
                    rel_ok += (min(d, 180 - d) < 15
                               and abs(r["wavelength_m"] - sp_m) / sp_m < 0.25)
    acc = rel_ok / max(n_rel, 1)
    ok3 = acc >= 0.8 and stuck / n < 0.1 and n_rel >= 0.25 * n
    ok_all &= ok3
    print(f"  {'PASS' if ok3 else 'FAIL'}  peaks stuck at crop scale: {stuck}/{n} "
          f"| flagged reliable: {n_rel}/{n} | reliable & correct: {rel_ok}/{n_rel} ({acc:.0%})")

    print("\n[4] Packet + strong monsoon swell (300 m, crop-filling) in the same crop")
    n = n_rel = rel_ok = swell_seen = 0
    for sp_m in [800, 1200, 1500]:
        for brg in np.linspace(0, 330, 12):
            img = make_synthetic_packet(700, 500, sp_m / PIXEL_SCALE_M, brg, rng,
                                        swell=(0.5, 300 / PIXEL_SCALE_M, brg + 40))
            r = extract_dominant_peak(img, PIXEL_SCALE_M)
            n += 1
            swell_seen += r["swell_snr"] >= SNR_RELIABLE
            if r["spectral_peak_snr"] >= SNR_RELIABLE and r["periods_in_crop"] >= PERIODS_RELIABLE:
                n_rel += 1
                d = abs(r["image_bearing_deg"] - brg) % 180
                rel_ok += (min(d, 180 - d) < 15
                           and abs(r["wavelength_m"] - sp_m) / sp_m < 0.25)
    acc = rel_ok / max(n_rel, 1)
    ok4 = acc >= 0.8 and swell_seen >= 0.8 * n and n_rel >= 0.25 * n
    ok_all &= ok4
    print(f"  {'PASS' if ok4 else 'FAIL'}  swell detected in diagnostic: {swell_seen}/{n} "
          f"| flagged reliable: {n_rel}/{n} | reliable & packet correct: {rel_ok}/{n_rel} ({acc:.0%})")

    print("\n[5] Red-heavy spectra (stronger, steeper background) -- harmonic check must")
    print("    not drag estimates to the long-wavelength limit (the v2 real-data failure)")
    n = harm = at_lim = n_rel = rel_ok = 0
    for sp_m in [800, 1200, 1500, 2000]:
        for brg in np.linspace(0, 330, 12):
            h, w = 700, 500
            img = make_synthetic_packet(h, w, sp_m / PIXEL_SCALE_M, brg, rng,
                                        red_amp=1.2, red_beta=2.6)
            r = extract_dominant_peak(img, PIXEL_SCALE_M)
            n += 1
            harm += r["harmonic_n"] > 1
            at_lim += r["wavelength_m"] > 0.9 * min(h, w) / MIN_PERIODS_IN_CROP * PIXEL_SCALE_M
            if r["spectral_peak_snr"] >= SNR_RELIABLE and r["periods_in_crop"] >= PERIODS_RELIABLE:
                n_rel += 1
                d = abs(r["image_bearing_deg"] - brg) % 180
                rel_ok += (min(d, 180 - d) < 15
                           and abs(r["wavelength_m"] - sp_m) / sp_m < 0.25)
    acc = rel_ok / max(n_rel, 1)
    ok5 = harm <= 0.15 * n and at_lim <= 0.15 * n and acc >= 0.8
    ok_all &= ok5
    print(f"  {'PASS' if ok5 else 'FAIL'}  harmonic jumps: {harm}/{n} | at long limit: {at_lim}/{n} "
          f"| reliable & correct: {rel_ok}/{n_rel} ({acc:.0%})")

    print("\n" + "=" * 70)
    print("ALL SELF-TESTS PASSED." if ok_all else "SELF-TEST FAILURES -- do not run on real data.")
    print("=" * 70)
    return ok_all


# ----------------------------------------------------------------------
# Real run
# ----------------------------------------------------------------------
def run():
    det_path, tiles_dir = Path(DETECTIONS_PARQUET), Path(TILES_DIR)
    for p, what in [(det_path, "DETECTIONS_PARQUET"), (tiles_dir, "TILES_DIR"),
                    (Path(SAFE_ROOT), "SAFE_ROOT")]:
        if not p.exists():
            print(f"ERROR: {what} not found: {p}")
            return

    det = pd.read_parquet(det_path)
    print(f"Loaded {len(det)} detections. Columns: {list(det.columns)}")
    print(f"Tile scale: {PIXEL_SCALE_M:.2f} m/px "
          f"({TILE_SIZE_KM} km tiles resampled to {OUTPUT_PIXELS}px)")

    # Land mask: older parquets were written before land-masking existed
    # (the original 798-detection run had ~450 detections on land). Always
    # re-apply it here so Tier 1 never measures rivers/deltas.
    if not Path(LAND_MASK_PATH).exists():
        print(f"ERROR: LAND_MASK_PATH not found: {LAND_MASK_PATH}")
        return
    is_on_land = load_land_checker(LAND_MASK_PATH)
    on_land = det.apply(lambda r: is_on_land(float(r["real_lat"]), float(r["real_lon"])), axis=1)
    print(f"Land mask: dropping {int(on_land.sum())} of {len(det)} detections on land")
    det = det[~on_land].reset_index(drop=True)

    # Tile filename: present in newer parquets, rebuilt from scene_id + tile_idx otherwise
    if "tile_filename" not in det.columns:
        lookup = build_tile_lookup(tiles_dir)
        print(f"No 'tile_filename' column -- matched via scene_id + tile_idx "
              f"against {len(lookup)} tiles in TILES_DIR")
        det["tile_filename"] = [lookup.get((s, int(t))) for s, t in
                                zip(det["scene_id"], det["tile_idx"])]

    tile_cache, transform_cache = {}, {}

    def get_transform(scene_id):
        if scene_id not in transform_cache:
            try:
                transform_cache[scene_id] = load_scene_transform(scene_id)
            except Exception as e:
                print(f"  WARNING: no GCP transform for {scene_id}: {e}")
                transform_cache[scene_id] = None
        return transform_cache[scene_id]

    # Whole-box coastline check. The centre-point check above misses boxes
    # whose centre is just offshore but which contain a coastline -- the
    # first real 'inspect' grid showed 6 of the top 12 'reliable' results
    # were coastlines. Here every point on a 9x9 grid over the box, grown
    # by COAST_BUFFER_KM, is checked against the land mask.
    touches = []
    for _, row in det.iterrows():
        fname, T = row["tile_filename"], get_transform(row["scene_id"])
        m = TILE_FILENAME_RE.match(fname) if fname else None
        if T is None or m is None:
            touches.append(False)          # can't place the box; centre check already done
            continue
        row0, col0 = int(m.group(3)), int(m.group(4))
        x1, y1, x2, y2 = get_box_xyxy(row)
        buf = COAST_BUFFER_KM * 1000 / NATIVE_PX_M          # scene pixels
        xs = np.linspace(col0 + x1 * TILE_TO_SCENE - buf, col0 + x2 * TILE_TO_SCENE + buf, 9)
        ys = np.linspace(row0 + y1 * TILE_TO_SCENE - buf, row0 + y2 * TILE_TO_SCENE + buf, 9)
        hit = False
        for sy in ys:
            for sx in xs:
                lon, lat = T * (sx, sy)
                if is_on_land(lat, lon):
                    hit = True
                    break
            if hit:
                break
        touches.append(hit)
    touches = np.array(touches, dtype=bool)
    print(f"Coastline check: dropping {int(touches.sum())} of {len(det)} detections whose box "
          f"(+{COAST_BUFFER_KM:g} km) contains land")
    det = det[~touches].reset_index(drop=True)

    results = []
    n_missing_tile = n_bad_crop = n_no_transform = n_low = 0

    for i, (_, row) in enumerate(det.iterrows()):
        fname = row["tile_filename"]
        if fname is None:
            n_missing_tile += 1
            continue
        if fname not in tile_cache:
            tp = tiles_dir / fname
            tile_cache[fname] = np.array(Image.open(tp).convert("L")) if tp.exists() else None
        tile = tile_cache[fname]
        if tile is None:
            n_missing_tile += 1
            continue

        x1, y1, x2, y2 = get_box_xyxy(row)
        crop = crop_box(tile, x1, y1, x2, y2, CROP_PAD_PX)
        peak = extract_dominant_peak(crop, PIXEL_SCALE_M)
        if peak is None:
            n_bad_crop += 1
            continue

        scene_id = row["scene_id"]
        T = get_transform(scene_id)

        if T is not None:
            geo_a, geo_b = geographic_bearing(
                T, row["scene_pixel_x"], row["scene_pixel_y"],
                peak["freq_y"], peak["freq_x"],
            )
        else:
            geo_a = geo_b = np.nan
            n_no_transform += 1

        low = (not np.isfinite(peak["spectral_peak_snr"])
               or peak["spectral_peak_snr"] < SNR_RELIABLE
               or peak["periods_in_crop"] < PERIODS_RELIABLE
               or T is None)
        n_low += int(low)

        results.append({
            "scene_id": scene_id,
            "tile_filename": fname,
            "box_x1": x1, "box_y1": y1, "box_x2": x2, "box_y2": y2,
            "real_lat": row.get("real_lat"),
            "real_lon": row.get("real_lon"),
            "confidence": row.get("confidence"),
            "crop_h": crop.shape[0], "crop_w": crop.shape[1],
            "wavelength_m": peak["wavelength_m"],
            "image_bearing_deg": peak["image_bearing_deg"],
            "bearing_deg_a": geo_a,
            "bearing_deg_b": geo_b,
            "spectral_peak_snr": peak["spectral_peak_snr"],
            "periods_in_crop": peak["periods_in_crop"],
            "harmonic_n": peak["harmonic_n"],
            "swell_snr": peak["swell_snr"],
            "swell_wavelength_m": peak["swell_wavelength_m"],
            "low_confidence": low,
        })
        if (i + 1) % 100 == 0:
            print(f"  processed {i + 1}/{len(det)}")

    out = pd.DataFrame(results)
    out.to_parquet(OUTPUT_PARQUET, index=False)

    print(f"\n{'=' * 60}\nTIER 1 SUMMARY\n{'=' * 60}")
    print(f"Detections in:              {len(det)}")
    print(f"Processed:                  {len(out)}")
    print(f"Tile file not found:        {n_missing_tile}")
    print(f"Crop too small/degenerate:  {n_bad_crop}")
    print(f"No GCP transform (bearing NaN): {n_no_transform}")
    print(f"Low-confidence flagged:     {n_low} ({100 * n_low / max(1, len(out)):.1f}%)"
          f"  [SNR < {SNR_RELIABLE} or < {PERIODS_RELIABLE} periods in crop]")
    if len(out):
        rel = out[~out["low_confidence"]]
        print(f"Reliable (go to Tier 2):    {len(rel)}")
        print("\nWavelength (m), reliable only:")
        print(rel["wavelength_m"].describe().round(1).to_string())
        print("\nWavelength (m), all:")
        print(out["wavelength_m"].describe().round(1).to_string())
        print("\nWhitened spectral SNR (all):")
        print(out["spectral_peak_snr"].describe().round(1).to_string())
        print("\nPeriods in crop (all):")
        print(out["periods_in_crop"].describe().round(1).to_string())
        print(f"\nHarmonic correction applied: {(out['harmonic_n'] > 1).sum()}")
        lim = out[["crop_h", "crop_w"]].min(axis=1) / MIN_PERIODS_IN_CROP * PIXEL_SCALE_M
        print(f"At long-wavelength search limit (>90%): {(out['wavelength_m'] > 0.9 * lim).sum()}")
        print(f"At 500 m search floor (<550 m):         {(out['wavelength_m'] < 550).sum()}")
        print(f"Swell present (100-500 m, SNR >= {SNR_RELIABLE:.0f}): "
              f"{(out['swell_snr'] >= SNR_RELIABLE).sum()}  "
              f"(diagnostic only -- not used for Tier 2)")
    print(f"\nSaved to {OUTPUT_PARQUET}")


REVIEW_N = 48            # detections to label by eye
REVIEW_SEED = 0          # fixed, so the same sample comes out every time
REVIEW_CSV = "/kaggle/input/datasets/mayankgupta005/manually-reviewed-detections/review_labels.csv" #Updated the path pointing towards uploaded modified csv
LABEL_HELP = {
    "W": "wave packet: several roughly parallel bands/stripes in open water",
    "C": "coast or land is the main feature",
    "S": "single line: ship wake, slick or a lone front",
    "T": "just texture/swell, no distinct feature",
    "U": "unsure",
}


def review():
    """Draw a RANDOM sample of detections (not the best-looking ones, so the
    result is an honest estimate) and save numbered picture sheets plus a
    CSV to fill in. The yellow box is what the detector found; the red line
    (reliable Tier 1 results only) is the estimated propagation direction."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    res = pd.read_parquet(OUTPUT_PARQUET).reset_index(drop=True)
    n = min(REVIEW_N, len(res))
    sample = res.sample(n=n, random_state=REVIEW_SEED).reset_index(drop=True)
    sample.insert(0, "review_id", range(1, n + 1))

    per_sheet, cols = 12, 4
    for s in range(0, n, per_sheet):
        chunk = sample.iloc[s:s + per_sheet]
        rows = int(np.ceil(len(chunk) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 4.4 * rows))
        for ax, (_, r) in zip(np.atleast_1d(axes).ravel(), chunk.iterrows()):
            tile = np.array(Image.open(Path(TILES_DIR) / r["tile_filename"]).convert("L"))
            # Show the box with generous context so the surroundings are visible
            pad = 120
            h, w = tile.shape
            X0 = max(0, int(r["box_x1"]) - pad); Y0 = max(0, int(r["box_y1"]) - pad)
            X1 = min(w, int(r["box_x2"]) + pad); Y1 = min(h, int(r["box_y2"]) + pad)
            ax.imshow(tile[Y0:Y1, X0:X1], cmap="gray")
            ax.add_patch(plt.Rectangle((r["box_x1"] - X0, r["box_y1"] - Y0),
                                       r["box_x2"] - r["box_x1"], r["box_y2"] - r["box_y1"],
                                       fill=False, ec="yellow", lw=1.5))
            if not r["low_confidence"]:
                cx = (r["box_x1"] + r["box_x2"]) / 2 - X0
                cy = (r["box_y1"] + r["box_y2"]) / 2 - Y0
                L = 0.35 * min(r["box_x2"] - r["box_x1"], r["box_y2"] - r["box_y1"])
                b = np.radians(r["image_bearing_deg"])
                ax.plot([cx - L * np.sin(b), cx + L * np.sin(b)],
                        [cy + L * np.cos(b), cy - L * np.cos(b)], "r-", lw=2)
            ax.set_title(f"#{r['review_id']}   ({r['real_lat']:.2f}N {r['real_lon']:.2f}E)",
                         fontsize=11, fontweight="bold")
            ax.axis("off")
        for ax in np.atleast_1d(axes).ravel()[len(chunk):]:
            ax.axis("off")
        fig.suptitle("Yellow = detector box.  Red line (when shown) = Tier 1 direction; "
                     "it should cross the stripes at right angles.", fontsize=11)
        fig.tight_layout()
        name = f"review_sheet_{s // per_sheet + 1}.png"
        fig.savefig(name, dpi=100)
        plt.close(fig)
        print(f"Saved {name}")

    out = sample[["review_id", "tile_filename", "real_lat", "real_lon", "confidence",
                  "low_confidence"]].copy()
    out["red_line_shown"] = ~out["low_confidence"]
    out = out.drop(columns="low_confidence")
    out["label"] = ""
    out["line_ok"] = ""
    out.to_csv(REVIEW_CSV, index=False)
    print(f"\nSaved {REVIEW_CSV} -- fill in two columns, then run MODE = 'score':")
    print("  label   (every row):")
    for k, v in LABEL_HELP.items():
        print(f"     {k} = {v}")
    print("  line_ok (only rows labelled W that show a red line):")
    print("     Y = red line crosses the stripes at roughly a right angle, N = it doesn't")


def _wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    hw = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - hw, c + hw)


def score():
    """Turn the filled-in review CSV into the numbers for the report."""
    df = pd.read_csv(REVIEW_CSV)
    df["label"] = df["label"].fillna("").astype(str).str.strip().str.upper()
    df["line_ok"] = df["line_ok"].fillna("").astype(str).str.strip().str.upper()
    bad = df[~df["label"].isin(LABEL_HELP.keys())]
    if len(bad):
        print(f"WARNING: {len(bad)} rows have no valid label (review_id "
              f"{bad['review_id'].tolist()[:10]}...) -- they are left out.")
    df = df[df["label"].isin(LABEL_HELP.keys())]
    decided = df[df["label"] != "U"]
    k, n = int((decided["label"] == "W").sum()), len(decided)
    lo, hi = _wilson(k, n)
    print("=" * 60)
    print("DETECTOR PRECISION ON OUR OWN SCENES (random sample, after land + coast filters)")
    print("=" * 60)
    for key, text in LABEL_HELP.items():
        print(f"  {key}: {int((df['label'] == key).sum()):3d}   {text}")
    if n:
        print(f"\n  Real wave packets: {k} of {n} decided = {k / n:.0%} "
              f"(95% range {lo:.0%} to {hi:.0%})")
        print("  -> report as: 'On a random sample of n=%d detections from our June 2021 "
              "scenes, %d%% (95%% CI %d-%d%%) were judged to be internal-wave packets.'"
              % (n, round(100 * k / n), round(100 * lo), round(100 * hi)))
    w = df[(df["label"] == "W") & df["red_line_shown"].astype(str).str.lower().eq("true")]
    w = w[w["line_ok"].isin(["Y", "N"])]
    if len(w):
        kk = int((w["line_ok"] == "Y").sum())
        a, b = _wilson(kk, len(w))
        print(f"\n  Tier 1 direction correct on real waves: {kk} of {len(w)} = "
              f"{kk / len(w):.0%} (95% range {a:.0%} to {b:.0%})")
    else:
        print("\n  No W rows with a red line and a Y/N in line_ok -- direction not scored.")


def main():
    # In a notebook, sys.argv holds Jupyter's own kernel args -- ignore them
    # and use MODE. From a terminal, a 'selftest'/'run' argument wins.
    in_notebook = "ipykernel" in sys.modules
    mode = MODE
    if not in_notebook and len(sys.argv) == 2 and sys.argv[1] in ("selftest", "run", "review", "score"):
        mode = sys.argv[1]
    print(f"Mode: {mode}")
    if mode == "selftest":
        selftest()
    elif mode == "run":
        run()
    elif mode == "review":
        review()
    elif mode == "score":
        score()
    else:
        print(f"Unknown MODE {mode!r}; use 'selftest', 'run', 'review' or 'score'.")


if __name__ == "__main__":
    main()

Mode: score
DETECTOR PRECISION ON OUR OWN SCENES (random sample, after land + coast filters)
  W:  14   wave packet: several roughly parallel bands/stripes in open water
  C:   3   coast or land is the main feature
  S:   8   single line: ship wake, slick or a lone front
  T:  13   just texture/swell, no distinct feature
  U:  10   unsure

  Real wave packets: 14 of 38 decided = 37% (95% range 23% to 53%)
  -> report as: 'On a random sample of n=38 detections from our June 2021 scenes, 37% (95% CI 23-53%) were judged to be internal-wave packets.'

  Tier 1 direction correct on real waves: 9 of 9 = 100% (95% range 70% to 100%)
